In [ ]:
import os, torch
print("working folder:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
%cd /content
!git clone https://github.com/samyuktaathreya/pinyin_audio_recognizer_model.git repo
%cd /content/repo/pinyin_scorer
%pip install -q pypinyin

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/pinyin_runs

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo log --oneline -1
!ls *.py
!grep -c "Hiss only" data.py

In [ ]:
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!unzip -q -o /content/drive/MyDrive/standard.zip -d /content/ext
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl
!python prepare_folders.py --root /content/ext/standard --out-dir manifests/folders

In [ ]:
!grep -c "Hiss only" data.py
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4 \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 300 --augment --workers 2

In [ ]:
!ps aux | grep "train.py" | grep -v grep | wc -l
!ls /content/repo/pinyin_scorer/manifests /content/ext 2>&1 | head -5
!ls /content/drive/MyDrive/pinyin_runs/stage4/last

In [ ]:
%cd /content/repo/pinyin_scorer

In [ ]:
!setsid nohup python -u train.py --init-from /content/drive/MyDrive/pinyin_runs/stage4/last \
    --train manifests/syl_train.jsonl manifests/folders/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage4b \
    --epochs 5 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 100 --augment --workers 2 \
    > /content/train_stage4b.log 2>&1 &

In [ ]:
!tail -3 /content/train_stage4b.log
!grep "^epoch [0-9]* initial" /content/train_stage4b.log

In [ ]:
!python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/stage4b/best \
    --manifests manifests/syl_test.jsonl manifests/folders/test_*.jsonl --out stage4_heldout.csv

In [ ]:
!ps aux | grep "train.py" | grep -v grep | grep -o "\-\-out [^ ]*" | sort -u

In [ ]:
import json
for line in open("/content/drive/MyDrive/pinyin_runs/stage4/log.jsonl"):
    d = json.loads(line)
    print(d["epoch"], *(f"{d[k]:.3f}" for k in
          ["syllable_initial_acc", "syllable_final_acc", "syllable_tone_acc", "syllable_all3_acc"]))

In [ ]:

%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!grep -c "no-augment-sources" prepare_folders.py; grep -c "duplicate_pairs" find_duplicates.py; grep -c "def speech_power" data.py

In [ ]:
!unzip -q -o /content/drive/MyDrive/standard.zip -d /content/std
!ls /content/std/standard

In [ ]:
ROOT = "/content/std/standard"
!python find_duplicates.py {ROOT}/*/

In [ ]:
import pandas as pd
from IPython.display import Audio, display
ex = pd.read_csv("duplicate_examples.csv")
pair = ex[(ex.source_a == "losttheory") & (ex.source_b == "uvm")]
for _, r in pair.sample(3, random_state=0).iterrows():
    print(f"\n{r.syllable}  (similarity {r.similarity})")
    display(Audio(r.file_a)); display(Audio(r.file_b))

In [ ]:
ROOT = "/content/std/standard"
!python prepare_folders.py --root {ROOT} --out-dir manifests/std \
    --test-sources gitcode studycli uvm --no-augment-sources uvm \
    --skip-sources youtube_man sreading losttheory
!ls manifests/syl_train.jsonl manifests/syl_test.jsonl

In [ ]:
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl

In [ ]:
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/std/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage5 \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 300 --augment --workers 2

In [ ]:
for name in ["stage2", "stage5"]:
    print(f"\n######## {name}")
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/std/test_*.jsonl --out {name}_std.csv
!cp stage2_std.csv stage5_std.csv /content/drive/MyDrive/

In [ ]:
import json, os, glob, collections
train = [json.loads(l) for f in ["manifests/syl_train.jsonl", "manifests/std/train.jsonl"] for l in open(f)]
per_speaker = collections.Counter(r["speaker"] for r in train)
print(len(train), "training clips:", dict(per_speaker))
train_paths = {os.path.realpath(r["audio"]) for r in train}
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    speakers = {r["speaker"] for r in rows}
    same_files = sum(os.path.realpath(r["audio"]) in train_paths for r in rows)
    print(f"{tf}: {len(rows)} clips, speakers {sorted(speakers)}, "
          f"speakers also in training: {sorted(speakers & set(per_speaker)) or 'none'}, "
          f"identical files in training: {same_files}")

In [ ]:
import re
from pathlib import Path
dst = Path("/content/tp_by_speaker")
for f in Path("/content/tone_perfect").rglob("*.mp3"):
    m = re.match(r"^[a-zü:v]+[1-5]_([A-Za-z0-9]+)", f.name)
    if m:
        d = dst / f"tp_{m.group(1)}"
        d.mkdir(parents=True, exist_ok=True)
        if not (d / f.name).exists():
            (d / f.name).symlink_to(f)
!python find_duplicates.py /content/tp_by_speaker/*/ /content/std/standard/*/ \
    --out tp_dup_examples.csv --summary tp_dup_pairs.csv

In [ ]:
import json, glob, random
from IPython.display import Audio, display
from score import Scorer
scorer = Scorer("/content/drive/MyDrive/pinyin_runs/stage5/best")
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    scored = []
    for r in random.Random(0).sample(rows, min(40, len(rows))):
        res = scorer.score(r["audio"], r["pinyin"])
        scored.append((min(p["score"] for p in res["parts"].values()), r, res["parts"]))
    scored.sort(key=lambda x: x[0])
    print(f"\n===== {tf}")
    for label, group in [("WORST", scored[:3]), ("BEST", scored[-3:])]:
        for conf, r, p in group:
            print(f"{label}: target {r['pinyin']:7} heard {p['initial']['heard']} + {p['final']['heard']} "
                  f"+ tone {p['tone']['heard']}   (weakest part {conf:.2f})")
            display(Audio(r["audio"]))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
# !grep -c "def normalize_loudness" data.py; grep -c "duplicate_pairs" find_duplicates.py

In [ ]:
for name in ["stage2", "stage5"]:
    print(f"\n######## {name}")
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/std/test_*.jsonl --out {name}_norm.csv

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!grep -c "duplicate_pairs" find_duplicates.py

In [ ]:
import json, os, glob, collections
train = [json.loads(l) for f in ["manifests/syl_train.jsonl", "manifests/std/train.jsonl"] for l in open(f)]
per_speaker = collections.Counter(r["speaker"] for r in train)
print(len(train), "training clips:", dict(per_speaker))
train_paths = {os.path.realpath(r["audio"]) for r in train}
for tf in ["manifests/syl_test.jsonl", *sorted(glob.glob("manifests/std/test_*.jsonl"))]:
    rows = [json.loads(l) for l in open(tf)]
    speakers = {r["speaker"] for r in rows}
    same_files = sum(os.path.realpath(r["audio"]) in train_paths for r in rows)
    print(f"{tf}: {len(rows)} clips, speakers {sorted(speakers)}, "
          f"speakers also in training: {sorted(speakers & set(per_speaker)) or 'none'}, "
          f"identical files in training: {same_files}")

In [ ]:
import re
from pathlib import Path
dst = Path("/content/tp_by_speaker")
for f in Path("/content/tone_perfect").rglob("*.mp3"):
    m = re.match(r"^[a-zü:v]+[1-5]_([A-Za-z0-9]+)", f.name)
    if m:
        d = dst / f"tp_{m.group(1)}"
        d.mkdir(parents=True, exist_ok=True)
        if not (d / f.name).exists():
            (d / f.name).symlink_to(f)
!python find_duplicates.py /content/tp_by_speaker/*/ /content/std/standard/*/ \
    --out tp_dup_examples.csv --summary tp_dup_pairs.csv

In [ ]:
import pandas as pd
from IPython.display import Audio, display

df = pd.read_csv("stage2_norm.csv")
g = df[df.source == "test_gitcode"]
print(len(g), "gitcode clips the model got wrong\n")
for _, r in g.sample(min(12, len(g)), random_state=1).iterrows():
    wrong = [h for h in ("initial", "final", "tone") if r[f"wrong_{h}"]]
    print(f"target {r.target:8} heard {r.heard_initial} + {r.heard_final} + tone {r.heard_tone}"
          f"   (wrong: {', '.join(wrong)})")
    display(Audio(r.audio))

In [ ]:
%cd /content/repo/pinyin_scorer

In [ ]:
import base64, io, random
import numpy as np, soundfile as sf
from IPython.display import HTML, display
from data import load_audio, SAMPLE_RATE

def clip_tag(path):
    """Audio player for one clip: volume evened out, silence trimmed off the ends."""
    wav = load_audio(path)
    n = len(wav) // 160
    rms = np.sqrt((wav[:n * 160].reshape(n, 160) ** 2).mean(axis=1))
    active = np.where(rms > 0.05 * rms.max())[0]
    if len(active):
        wav = wav[max(0, active[0] * 160 - 3200): (active[-1] + 1) * 160 + 3200]
    buf = io.BytesIO()
    sf.write(buf, wav, SAMPLE_RATE, format="WAV", subtype="PCM_16")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<audio controls preload="none" style="width:110px" src="data:audio/wav;base64,{b64}"></audio>'

def grid(kind="final", speakers=None):
    idx, units = (1, FINALS) if kind == "final" else (0, INITIALS)
    speakers = speakers or sorted(by_speaker)
    rng = random.Random(0)
    head = "<tr><th></th>" + "".join(f"<th>{s}</th>" for s in speakers) + "</tr>"
    rows = []
    for unit in units:
        cells = [f"<th>{unit}</th>"]
        for s in speakers:
            cands = [(p, r) for p, r in by_speaker[s] if p[idx] == unit]
            if not cands:
                cells.append("<td>–</td>")
                continue
            p, r = rng.choice([c for c in cands if c[0][2] == "1"] or cands)
            cells.append(f"<td style='font-size:11px'>{r['pinyin']}<br>{clip_tag(r['audio'])}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f"<div style='overflow-x:auto'><table>{head}{''.join(rows)}</table></div>"))

web = ["davinfifield", "digmandarin", "gitcode", "hotoo", "hugolpz", "mandarintemple",
       "misc", "studycli", "uvm", "yabla", "yoyochinese"]
grid("final", web)

In [ ]:
grid("initial", web)

In [ ]:
checks = [("digmandarin", "lan"), ("digmandarin", "shang"), ("digmandarin", "zhong"),
          ("hugolpz", "nao"), ("hugolpz", "fan"), ("hugolpz", "meng"),
          ("gitcode", "gong"), ("davinfifield", "miao"), ("davinfifield", "weng")]
for spk, base in checks:
    for p, r in by_speaker[spk]:
        if r["pinyin"][:-1] == base:
            print(f"{spk:14} {r['pinyin']:8} {r['audio'].split('/')[-1]}")
            display(Audio(r["audio"]))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull
!python prepare_folders.py --root /content/std/standard --out-dir manifests/good \
    --test-sources studycli uvm --no-augment-sources uvm \
    --skip-sources youtube_man sreading losttheory davinfifield digmandarin gitcode hugolpz yoyochinese

In [ ]:
%cd /content/repo/pinyin_scorer

In [ ]:
import json, glob, random
import soundfile as sf
from IPython.display import Audio, display

rows = [json.loads(l) for m in ["manifests/good/train.jsonl", *glob.glob("manifests/good/test_*.jsonl")]
        for l in open(m)]
speakers = ["hotoo", "mandarintemple", "misc", "studycli", "uvm", "yabla"]
rng = random.Random()          # different clips every run; use random.Random(0) to repeat a set

for spk in speakers:
    print(f"\n==================== {spk}")
    mine = [r for r in rows if r["speaker"] == spk]
    for tone in "1234":
        pool = [r for r in mine if r["pinyin"].endswith(tone)]
        for r in rng.sample(pool, min(2, len(pool))):
            wav, sr = sf.read(r["audio"])
            print(f"{r['pinyin']:8} {r['audio'].split('/')[-1]}")
            display(Audio(wav.T if wav.ndim > 1 else wav, rate=sr))

In [ ]:
import json, glob, random, base64, io, collections
import soundfile as sf
from IPython.display import HTML, display
from pinyin_units import split_syllable, INITIALS, FINALS
from data import load_audio, SAMPLE_RATE

by_speaker = collections.defaultdict(list)
for m in ["manifests/good/train.jsonl", *glob.glob("manifests/good/test_*.jsonl")]:
    for line in open(m):
        r = json.loads(line)
        parts = split_syllable(r["pinyin"])
        if parts:
            by_speaker[r["speaker"]].append((parts, r))

def clip_tag(path):
    buf = io.BytesIO()
    sf.write(buf, load_audio(path), SAMPLE_RATE, format="WAV", subtype="PCM_16")
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<audio controls preload="none" style="width:110px" src="data:audio/wav;base64,{b64}"></audio>'

def grid(kind="initial", speakers=None, seed=None):
    idx, units = (0, INITIALS) if kind == "initial" else (1, FINALS)
    speakers = speakers or sorted(by_speaker)
    rng = random.Random(seed)
    head = "<tr><th></th>" + "".join(f"<th>{s}</th>" for s in speakers) + "</tr>"
    rows = []
    for unit in units:
        cells = [f"<th>{unit}</th>"]
        for s in speakers:
            cands = [r for p, r in by_speaker[s] if p[idx] == unit]
            if not cands:
                cells.append("<td>–</td>")
                continue
            r = rng.choice(cands)
            cells.append(f"<td style='font-size:11px'>{r['pinyin']}<br>{clip_tag(r['audio'])}</td>")
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f"<div style='overflow-x:auto'><table>{head}{''.join(rows)}</table></div>"))

grid("initial", ["hotoo", "mandarintemple", "misc", "studycli", "uvm", "yabla"])

In [ ]:
!git -C /content/repo pull

In [ ]:
!python dump_probs.py --model /content/drive/MyDrive/pinyin_runs/stage2/best \
    --manifests manifests/syl_test.jsonl manifests/good/train.jsonl manifests/good/test_*.jsonl \
    --out /content/drive/MyDrive/pinyin_runs/stage2_probs.csv

In [ ]:
import pandas as pd
from IPython.display import Audio, display

CSV = "/content/drive/MyDrive/pinyin_runs/stage2_probs.csv"
SOURCES = ["hotoo", "misc"]
SYLLABLES = ["zhi", "chi", "shi", "ri", "zi", "ci", "si"]

df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
df = df[df.source.isin(SOURCES) & df.target.str[:-1].isin(SYLLABLES)].copy()

def heard(row, part):   # the model's top guess for one part, ignoring silence
    cols = [c for c in df.columns if c.startswith(part + ":") and "silence" not in c]
    return max(cols, key=lambda c: float(row[c])).split(":")[1]

for part in ["initial", "final", "tone"]:
    df["heard_" + part] = df.apply(heard, axis=1, part=part)
df["wrong"] = (df.heard_initial != df.target_initial) | (df.heard_final != df.target_final)
df = df.sort_values(["wrong", "source", "target"], ascending=[False, True, True])

print(f"{len(df)} clips, {df.wrong.sum()} where the model heard a different initial or final\n")
for _, r in df.iterrows():
    mark = "WRONG" if r.wrong else "ok   "
    print(f"{mark}  {r.source:6} {r.target:5}  model heard: {r.heard_initial} + {r.heard_final} + tone {r.heard_tone}")
    display(Audio(r.audio))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull

In [ ]:
# Run A: clean data only
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/good/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage6a \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 200 --augment --workers 2

In [ ]:
# Run B: the same, plus the two changes
!python train.py --init-from /content/drive/MyDrive/pinyin_runs/stage2/best \
    --train manifests/syl_train.jsonl manifests/good/train.jsonl \
    --dev manifests/syl_test.jsonl --out /content/drive/MyDrive/pinyin_runs/stage6b \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 200 --augment --workers 2 \
    --voice-augment --per-head-mix

In [ ]:
for name in ["stage2", "stage6a", "stage6b"]:
    !python eval_sources.py --model /content/drive/MyDrive/pinyin_runs/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl --out {name}_good.csv

In [ ]:
!python dump_probs.py --model /content/drive/MyDrive/pinyin_runs/stage6b/best \
    --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl \
    --out /content/drive/MyDrive/pinyin_runs/stage6b_probs.csv

In [ ]:
import pandas as pd
from IPython.display import Audio, display

CSV = "/content/drive/MyDrive/pinyin_runs/stage6b_probs.csv"
SOURCE = "uvm"       # "uvm", "studycli", "FV3", "MV3", or None for all
PART   = "final"     # "initial", "final" or "tone"
TARGET = "eng"       # what the clip is labelled
HEARD  = "ang"       # what the model heard instead, or None for any wrong answer
LIMIT  = 30          # most clips to play

df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
cols = [c for c in df.columns if c.startswith(PART + ":") and "silence" not in c]
probs = df[cols].astype(float)
df["heard"] = probs.idxmax(axis=1).str.split(":").str[1]
df["p_target"] = [2.718281828 ** probs.at[i, f"{PART}:{t}"] for i, t in zip(df.index, df["target_" + PART])]

m = (df["target_" + PART] == TARGET) & (df.heard != TARGET)
if HEARD:  m &= df.heard == HEARD
if SOURCE: m &= df.source == SOURCE
hits = df[m].sort_values("p_target")

print(f"{len(hits)} clips labelled {TARGET} where the model heard {HEARD or 'something else'}\n")
for _, r in hits.head(LIMIT).iterrows():
    print(f"{r.source:9} {r.target:8} heard {r.heard:5} target's probability {r.p_target:.1%}")
    display(Audio(r.audio))

In [ ]:
%cd /content/repo/pinyin_scorer
!git -C /content/repo pull

In [ ]:
# ---- Session setup: run once at the start of every Colab session (T4 runtime) ----
REPO_URL = "https://github.com/samyuktaathreya/pinyin_audio_recognizer_model.git"   # <- put your repo's address here

from google.colab import drive
drive.mount('/content/drive')

# Code: clone the first time, otherwise pull the latest push
%cd /content
!git clone -q {REPO_URL} repo 2>/dev/null || git -C repo pull -q
%cd /content/repo/pinyin_scorer
%pip install -q pypinyin librosa
%pip install -q praat-parselmouth

# Audio: Colab's disk is wiped every session, so unpack it again
!unzip -q -o /content/drive/MyDrive/tone_perfect.zip -d /content/tone_perfect
!unzip -q -o /content/drive/MyDrive/standard.zip -d /content/std

# Manifests (lists of clips with their labels)
!python prepare_syllables.py --audio-dir /content/tone_perfect --test-speakers MV3 FV3 \
    --out-train manifests/syl_train.jsonl --out-test manifests/syl_test.jsonl
!python prepare_folders.py --root /content/std/standard --out-dir manifests/good \
    --test-sources studycli uvm --no-augment-sources uvm \
    --skip-sources youtube_man sreading losttheory davinfifield digmandarin gitcode hugolpz yoyochinese

# Checks: every line should end in True, and the GPU line should name a T4
import os, glob, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4")
print("new training code:", "--sentences" in open("train.py").read())
print("threshold rule in score.py:", "DEFAULT_THRESHOLD" in open("score.py").read())
print("Tone Perfect audio:", os.path.exists("/content/tone_perfect"))
print("web audio:", os.path.exists("/content/std/standard/uvm"))
for m in ["manifests/syl_train.jsonl", "manifests/syl_test.jsonl", "manifests/good/train.jsonl"] + sorted(glob.glob("manifests/good/test_*.jsonl")):
    print(f"{m}: {sum(1 for _ in open(m))} clips")

In [ ]:
runs = "/content/drive/MyDrive/pinyin_runs"
candidates = {"probe_w2v_base": "TencentGameMate/chinese-wav2vec2-base",
              "probe_hubert_base": "TencentGameMate/chinese-hubert-base",
              "probe_hubert_large": "TencentGameMate/chinese-hubert-large"}
for name, enc in candidates.items():
    !python train.py --encoder {enc} --freeze-encoder --per-head-mix \
        --train manifests/syl_train.jsonl manifests/good/train.jsonl \
        --dev manifests/syl_test.jsonl --out {runs}/{name} \
        --epochs 10 --head-lr 2e-3 --warmup-steps 100 --workers 2
for name in candidates:
    !python eval_sources.py --model {runs}/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl --out {name}.csv

In [ ]:
# ---- AISHELL setup (needed every session, because Colab's disk is wiped) ----
import os, glob
if not os.path.exists("/content/data_aishell/wav/train"):
    !tar -xzf /content/drive/MyDrive/data_aishell.tgz -C /content
    # AISHELL ships one archive per speaker inside wav/; unpack them, then delete the archives
    !cd /content/data_aishell/wav && for f in *.tar.gz; do tar -xzf "$f" && rm "$f"; done
print("speakers unpacked:", len(glob.glob("/content/data_aishell/wav/*/S*")))

In [ ]:
!python prepare_aishell.py --aishell-dir /content/data_aishell --split train --hours 15 --out manifests/aishell_train.jsonl
!python prepare_aishell.py --aishell-dir /content/data_aishell --split dev --hours 1 --out manifests/aishell_dev.jsonl

# ---- Stage 1 on HuBERT ----
!python train.py --encoder TencentGameMate/chinese-hubert-base \
    --train manifests/aishell_train.jsonl --dev manifests/aishell_dev.jsonl \
    --out /content/drive/MyDrive/pinyin_runs/stage1_hubert \
    --epochs 11 --per-head-mix --workers 2

In [ ]:
import json
for name in ["stage1/best", "stage1_hubert/best", "stage7_hubert/best"]:
    try:
        cfg = json.load(open(f"/content/drive/MyDrive/pinyin_runs/{name}/encoder/config.json"))
        print(f"{name:20} {cfg['model_type']}")
    except FileNotFoundError:
        print(f"{name:20} not trained yet")

In [ ]:
runs = "/content/drive/MyDrive/pinyin_runs"

# Stage 7: HuBERT, syllables with sentences mixed in
!python train.py --init-from {runs}/stage1_hubert/best \
    --train manifests/syl_train.jsonl manifests/good/train.jsonl \
    --sentences manifests/aishell_train.jsonl --sentence-hours 3 \
    --dev manifests/syl_test.jsonl --out {runs}/stage7_hubert \
    --epochs 8 --lr 2e-5 --head-lr 2e-4 --warmup-steps 200 \
    --augment --voice-augment --per-head-mix --workers 2

# Compare against the earlier models on the unseen voices
for name in ["stage2", "stage6b", "stage7_hubert"]:
    !python eval_sources.py --model {runs}/{name}/best \
        --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl --out {name}_good.csv

# Save every clip's probabilities, for the threshold analysis
!python dump_probs.py --model {runs}/stage7_hubert/best \
    --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl \
    --out {runs}/stage7_hubert_probs.csv

In [ ]:
import pandas as pd, numpy as np, librosa, matplotlib.pyplot as plt

CSV = "/content/drive/MyDrive/pinyin_runs/stage7_hubert_probs.csv"
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
tone_cols = [c for c in df.columns if c.startswith("tone:") and "silence" not in c]
df["heard_tone"] = df[tone_cols].astype(float).idxmax(axis=1).str.split(":").str[1]

groups = {
    "studycli tone 2, heard as 3": (df.source == "studycli") & (df.target_tone == "2") & (df.heard_tone == "3"),
    "studycli tone 2, heard as 2": (df.source == "studycli") & (df.target_tone == "2") & (df.heard_tone == "2"),
    "studycli tone 3":             (df.source == "studycli") & (df.target_tone == "3"),
    "FV3 tone 2 (reference)":      (df.source == "FV3") & (df.target_tone == "2"),
}

def contour(path):
    y, sr = librosa.load(path, sr=16000)
    f0, voiced, _ = librosa.pyin(y, fmin=60, fmax=500, sr=sr, frame_length=1024)
    f0 = f0[voiced]
    return f0, len(f0) * 256 / sr        # pitch values, seconds of voicing

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, (name, mask) in zip(axes, groups.items()):
    clips = df[mask].sample(min(15, mask.sum()), random_state=0)
    curves = [contour(p) for p in clips.audio]
    ref = np.median(np.concatenate([c for c, _ in curves if len(c)]))   # speaker's typical pitch
    for f0, _ in curves:
        if len(f0) > 5:
            ax.plot(np.linspace(0, 1, len(f0)), 12 * np.log2(f0 / ref), alpha=0.6)
    ax.set_title(f"{name}\nvoiced {np.mean([s for _, s in curves]):.2f} s on average")
    ax.set_xlabel("position in syllable"); ax.axhline(0, color="gray", lw=0.5)
axes[0].set_ylabel("semitones from speaker's median")
plt.tight_layout(); plt.show()

In [ ]:
%pip install -q praat-parselmouth

In [ ]:
import numpy as np, pandas as pd, parselmouth

CSV = "/content/drive/MyDrive/pinyin_runs/stage2_probs.csv"
SPEAKERS = ["uvm", "studycli", "hotoo", "mandarintemple", "yabla", "FV3", "MV3"]
PAIRS = [("eng", "ang"), ("en", "an"), ("ou", "ao")]

df = pd.read_csv(CSV, dtype=str, keep_default_na=False)

def vowel_f1(path):
    """F1 (Hz) early in the vowel, before the nasal or second vowel takes over."""
    snd = parselmouth.Sound(path)
    pitch = snd.to_pitch()
    voiced = pitch.xs()[pitch.selected_array["frequency"] > 0]
    if len(voiced) < 5:
        return np.nan
    t0, t1 = voiced[0], voiced[-1]
    formant = snd.to_formant_burg(maximum_formant=5500)
    times = t0 + (t1 - t0) * np.linspace(0.2, 0.4, 5)
    return np.nanmedian([formant.get_value_at_time(1, t) for t in times])

rows = []
for speaker in SPEAKERS:
    for closed, open_ in PAIRS:
        f1 = {}
        for final in (closed, open_):
            clips = df[(df.source == speaker) & (df.target_final == final)].audio
            f1[final] = np.nanmedian([vowel_f1(p) for p in clips])
        rows.append({"speaker": speaker, "pair": f"{closed}/{open_}",
                     "F1 closed": f1[closed], "F1 open": f1[open_],
                     "closed as % of open": 100 * f1[closed] / f1[open_]})
print(pd.DataFrame(rows).round(0).to_string(index=False))

In [ ]:
import numpy as np, pandas as pd
from scipy.signal import resample_poly
from data import load_audio
from score import Scorer

MODEL = "/content/drive/MyDrive/pinyin_runs/stage6b/best"
CSV = "/content/drive/MyDrive/pinyin_runs/stage2_probs.csv"
FACTORS = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10]    # below 1 lowers every frequency, above 1 raises it
GROUPS = [("FV3", "eng"), ("FV3", "ang"), ("MV3", "eng"), ("MV3", "ang"),
          ("yabla", "eng"), ("yabla", "ang")]

scorer = Scorer(MODEL)
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)

def shifted(path, factor):
    """Scale every frequency in the clip (pitch and vowel resonances) by `factor`."""
    wav = load_audio(path)
    return resample_poly(wav, 1000, int(round(1000 * factor))).astype(np.float32)

rows = []
for speaker, final in GROUPS:
    clips = df[(df.source == speaker) & (df.target_final == final)]
    row = {"clips": f"{speaker} {final} ({len(clips)})"}
    for factor in FACTORS:
        heard = [scorer.score(shifted(r.audio, factor), r.target)["parts"]["final"]["heard_unit"]
                 for r in clips.itertuples()]
        row[f"x{factor:.2f}"] = f"{100 * np.mean([h == final for h in heard]):.0f}%"
    rows.append(row)
    print(row, flush=True)

print("\nShare of clips whose final is heard correctly, at each frequency shift:")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
import numpy as np, pandas as pd
from collections import Counter
from scipy.signal import resample_poly
from data import load_audio
from score import Scorer

MODEL = "/content/drive/MyDrive/pinyin_runs/stage6b/best"
CSV = "/content/drive/MyDrive/pinyin_runs/stage2_probs.csv"
FACTORS = [0.85, 0.90, 1.00]
GROUPS = [("FV3", "ang"), ("FV3", "eng")]

scorer = Scorer(MODEL)
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)

def shifted(path, factor):
    wav = load_audio(path)
    return resample_poly(wav, 1000, int(round(1000 * factor))).astype(np.float32)

for speaker, final in GROUPS:
    clips = df[(df.source == speaker) & (df.target_final == final)]
    for factor in FACTORS:
        heard = Counter(scorer.score(shifted(r.audio, factor), r.target)["parts"]["final"]["heard_unit"]
                        for r in clips.itertuples())
        print(f"{speaker} {final} at x{factor:.2f}: " + ", ".join(f"{u} {n}" for u, n in heard.most_common()))

In [ ]:
!python shift_test.py --model {runs}/stage6b/best \
    --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl \
    --sources FV3 MV3 studycli --out {runs}/shift_6b.csv --check

In [ ]:
!python shift_test.py --model {runs}/stage6b/best \
    --manifests manifests/syl_test.jsonl manifests/good/test_*.jsonl \
    --sources FV3 MV3 studycli --out {runs}/shift_6b.csv

In [ ]:
%pip install -q praat-parselmouth
!python train.py --init-from {runs}/stage2/best \
    --train manifests/syl_train.jsonl manifests/good/train.jsonl \
    --dev manifests/syl_test.jsonl --out {runs}/stage8_praat \
    --epochs 6 --lr 1.5e-5 --head-lr 1.5e-4 --warmup-steps 200 --augment --workers 2 \
    --praat-augment --per-head-mix

In [ ]:

runs = "/content/drive/MyDrive/pinyin_runs"
TEST = "manifests/syl_test.jsonl manifests/good/test_*.jsonl"

# top-guess accuracy and most common mistakes, both models
for name in ["stage6b", "stage8_praat"]:
    !python eval_sources.py --model {runs}/{name}/best --manifests {TEST} --out {name}_good.csv

# full probabilities, for false alarms
!python dump_probs.py --model {runs}/stage8_praat/best --manifests {TEST} \
    --out {runs}/stage8_praat_probs.csv

# shift test, same voices as stage 6b's run
!python shift_test.py --model {runs}/stage8_praat/best --manifests {TEST} \
    --sources FV3 MV3 studycli --out {runs}/shift_8.csv

In [ ]:
import pandas as pd, numpy as np
runs = "/content/drive/MyDrive/pinyin_runs"
THR = 0.05
MODELS = {"6b": "stage6b", "8": "stage8_praat"}

def load(name):
    df = pd.read_csv(f"{runs}/{name}_probs.csv", dtype=str, keep_default_na=False)
    for part in ["initial", "final", "tone"]:
        cols = [c for c in df.columns if c.startswith(part + ":")]
        P = np.exp(df[cols].astype(float))
        df[f"p_{part}"] = [P.at[i, f"{part}:{t}"] for i, t in zip(df.index, df["target_" + part])]
        df[f"heard_{part}"] = P[[c for c in cols if "silence" not in c]].idxmax(axis=1).str.split(":").str[1]
        df[f"P_{part}"] = list(P.to_dict("records"))
    df["flagged"] = (df[["p_initial", "p_final", "p_tone"]] < THR).any(axis=1)
    df["all3"] = np.logical_and.reduce([df[f"heard_{p}"] == df[f"target_{p}"] for p in ["initial", "final", "tone"]])
    df["group"] = df.source.where(df.source.isin(["FV3", "MV3"]), "web: " + df.source)
    return df

dfs = {k: load(v) for k, v in MODELS.items()}

# --- accuracy and false alarms per voice
rows = []
for k, df in dfs.items():
    for g, d in list(df.groupby("group")) + [("BOTH WEB VOICES", df[df.group.str.startswith("web")])]:
        rows.append({"model": k, "voice": g, "clips": len(d),
                     "all-3 accuracy %": round(100 * d.all3.mean(), 1),
                     "false alarms %": round(100 * d.flagged.mean(), 1)})
print(pd.DataFrame(rows).pivot(index="voice", columns="model", values=["all-3 accuracy %", "false alarms %"]), "\n")

# --- error detection must not get worse: native clip of SAID scored as an attempt at OTHER
pairs = [("ang", "eng"), ("eng", "ang"), ("an", "en"), ("en", "an"), ("ou", "ong"), ("ao", "ou")]
rows = []
for said, attempt in pairs:
    r = {"clip says": said, "scored as attempt at": attempt}
    for k, df in dfs.items():
        d = df[df.target_final == said]
        r[f"{k} missed %"] = round(100 * np.mean([p[f"final:{attempt}"] >= THR for p in d.P_final]), 1)
    rows.append(r)
print("swapped-target misses (lower is better):")
print(pd.DataFrame(rows).to_string(index=False), "\n")

# --- shift test: robustness score and the key columns
FINALS = ["eng", "ong", "ou", "en", "in", "ao", "ang"]
REALISTIC = ["F0.90", "F1.10", "P-2", "P+2", "B0.90", "B1.10"]
SHOW = ["praat_1.00", "F0.90", "F1.10", "F1.15", "P-4", "P-2", "B0.90", "B1.10"]
sums = {k: pd.read_csv(f"{runs}/shift_{k}_summary.csv") for k in MODELS}
for k, s in sums.items():
    a = s[(s.source == "ALL") & s.final.isin(FINALS)]
    w = lambda c: np.average(a[a.condition.isin(c)].final_acc, weights=a[a.condition.isin(c)].n)
    print(f"stage {k}: baseline {w(['praat_1.00']):.1f}%   realistic shifts {w(REALISTIC):.1f}%   "
          f"gap {w(['praat_1.00']) - w(REALISTIC):.1f} points")
for src in ["ALL", "studycli", "MV3", "FV3"]:
    t = pd.concat({k: s[(s.source == src) & s.final.isin(FINALS)]
                   .pivot(index="final", columns="condition", values="final_acc")[SHOW]
                   for k, s in sums.items()}, axis=1).swaplevel(axis=1).sort_index(axis=1, level=0)
    print(f"\n=== {src}: final accuracy %, 6b vs 8 for each condition")
    print(t[SHOW].round(0).to_string())

In [ ]:
import pandas as pd, numpy as np
runs = "/content/drive/MyDrive/pinyin_runs"
THR = 0.05
PARTS = ["initial", "final", "tone"]

def P(name):
    df = pd.read_csv(f"{runs}/{name}_probs.csv", dtype=str, keep_default_na=False).set_index("audio")
    unit = [c for c in df.columns if ":" in c]
    df[unit] = np.exp(df[unit].astype(float))
    return df

base = {"6b": P("stage6b"), "7": P("stage7_hubert"), "8": P("stage8_praat")}

def combo(keys):
    common = base[keys[0]].index
    for k in keys[1:]:
        common = common.intersection(base[k].index)
    d = base[keys[0]].loc[common].copy()
    unit = [c for c in d.columns if ":" in c]
    d[unit] = sum(base[k].loc[common, unit] for k in keys) / len(keys)
    return d

def score(d):
    out = pd.DataFrame({"source": d.source}, index=d.index)
    for part in PARTS:
        cols = [c for c in d.columns if c.startswith(part + ":")]
        units = [c for c in cols if "silence" not in c]
        out[f"target_{part}"] = d["target_" + part]
        out[f"heard_{part}"] = d[units].idxmax(axis=1).str.split(":").str[1]
        out[f"ok_{part}"] = out[f"heard_{part}"] == out[f"target_{part}"]
        idx = [cols.index(f"{part}:{t}") for t in d["target_" + part]]
        out[f"p_{part}"] = d[cols].to_numpy()[np.arange(len(d)), idx]
    out["all3"] = out[[f"ok_{p}" for p in PARTS]].all(axis=1)
    out["flag"] = (out[[f"p_{p}" for p in PARTS]] < THR).any(axis=1)
    return out

# 1. accuracy per part, per voice
s6, s8 = score(base["6b"]), score(base["8"])
rows = []
for src in ["FV3", "MV3", "studycli", "uvm"]:
    for part in PARTS:
        a6 = s6[s6.source == src][f"ok_{part}"].mean()
        a8 = s8[s8.source == src][f"ok_{part}"].mean()
        rows.append({"voice": src, "part": part, "6b %": round(100 * a6, 1),
                     "8 %": round(100 * a8, 1), "change": round(100 * (a8 - a6), 1)})
print(pd.DataFrame(rows).to_string(index=False), "\n")

# 2. studycli tone confusions
for k, s in [("6b", s6), ("8", s8)]:
    d = s[(s.source == "studycli") & ~s.ok_tone]
    print(f"stage {k} studycli tone errors: {len(d)};  most common (target->heard):",
          d.groupby(["target_tone", "heard_tone"]).size().sort_values(ascending=False).head(4).to_dict())
print()

# 3. combinations, on the two web voices
rows = []
for keys in [["6b"], ["7"], ["8"], ["6b", "7"], ["6b", "8"], ["7", "8"], ["6b", "7", "8"]]:
    s = score(combo(keys))
    w = s[s.source.isin(["studycli", "uvm"])]
    rows.append({"models": " + ".join(keys), "web all-3 %": round(100 * w.all3.mean(), 1),
                 "web false alarms %": round(100 * w.flag.mean(), 1),
                 "studycli %": round(100 * w[w.source == "studycli"].all3.mean(), 1),
                 "uvm %": round(100 * w[w.source == "uvm"].all3.mean(), 1)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
import pandas as pd, numpy as np
runs = "/content/drive/MyDrive/pinyin_runs"

def heard(name):
    df = pd.read_csv(f"{runs}/{name}_probs.csv", dtype=str, keep_default_na=False).set_index("audio")
    cols = [c for c in df.columns if c.startswith("final:") and "silence" not in c]
    P = np.exp(df[cols].astype(float))
    return df[["source", "target_final"]], P

def table(name, P, df):
    h = P.idxmax(axis=1).str.split(":").str[1]
    rows = []
    for src in ["FV3", "MV3", "studycli", "uvm"]:
        r = {"model": name, "voice": src}
        for said, other in [("eng", "ang"), ("ang", "eng"), ("en", "an"), ("an", "en")]:
            m = (df.source == src) & (df.target_final == said)
            r[f"{said}->{other}"] = f"{(h[m] == other).sum()}"
        rows.append(r)
    return rows

d6, P6 = heard("stage6b"); d7, P7 = heard("stage7_hubert"); d8, P8 = heard("stage8_praat")
common = d6.index.intersection(d7.index)
rows = table("6b", P6, d6) + table("8", P8, d8) + \
       table("6b+7", (P6.loc[common] + P7.loc[common]) / 2, d6.loc[common])
print(pd.DataFrame(rows).set_index(["voice", "model"]).sort_index().to_string())

In [ ]:
import pandas as pd, numpy as np
runs = "/content/drive/MyDrive/pinyin_runs"
df = pd.read_csv(f"{runs}/stage2_probs.csv", dtype=str, keep_default_na=False)
cols = [c for c in df.columns if c.startswith("final:") and "silence" not in c]
Pf = np.exp(df[cols].astype(float))
df["heard"] = Pf.idxmax(axis=1).str.split(":").str[1]
TRAINED_ON = {"FV1", "FV2", "MV1", "MV2"}   # stage 2's own training voices: numbers will look too good

rows = []
for src, d in df.groupby("source"):
    r = {"voice": src + (" (trained)" if src in TRAINED_ON else "")}
    for fin in ["eng", "en", "ang", "an", "ong", "ou"]:
        x = d[d.target_final == fin]
        if len(x):
            wrong = x[x.heard != fin].heard.value_counts()
            r[fin] = f"{100 * (x.heard == fin).mean():.0f}%" + (f" ({wrong.index[0]})" if len(wrong) else "")
    rows.append(r)
print("stage 2, final accuracy per voice (most common wrong answer in brackets)")
print(pd.DataFrame(rows).set_index("voice").to_string())

In [ ]:
import json, math, numpy as np
from IPython.display import Audio, display
import shift_test, data
from data import SAMPLE_RATE, load_audio, normalize_loudness, pad_audio
from score import Scorer
shift_test._worker_init()          # loads parselmouth / librosa for make_versions

runs = "/content/drive/MyDrive/pinyin_runs"
scorers = {"6b": Scorer(f"{runs}/stage6b/best"), "8": Scorer(f"{runs}/stage8_praat/best")}
TEST_M  = ["manifests/syl_test.jsonl", "manifests/good/test_studycli.jsonl", "manifests/good/test_uvm.jsonl"]
TRAIN_M = ["manifests/syl_train.jsonl", "manifests/good/train.jsonl"]

def find(manifests, speaker, pinyin):
    for m in manifests:
        for line in open(m):
            r = json.loads(line)
            if (r.get("speaker") or r.get("source")) == speaker and r["pinyin"] == pinyin:
                r["source"] = speaker
                return r
    print(f"  (no {pinyin} clip from {speaker})")

def play(wav, pinyin, label):
    heard = []
    for k, s in scorers.items():
        p = shift_test.get_probs(s, wav, pinyin)["final"]
        tgt = shift_test.top_unit(p)
        want = dict(zip(shift_test.PARTS, __import__("pinyin_units").split_syllable(pinyin)))["final"]
        heard.append(f"{k}: heard {tgt} (p {want}={p[want]:.2f})")
    print(f"{label:34} " + "   ".join(heard))
    display(Audio(wav, rate=SAMPLE_RATE))

# ---------------------------------------------------------- 1. what the model was TESTED on
TEST_CLIPS = [("studycli", "feng1"), ("uvm", "feng1"), ("MV3", "feng1"), ("FV3", "fang1")]
TEST_CONDS = ["praat_1.00", "F1.10", "F1.15", "P-4", "F0.90", "B1.10"]
conds = [c for c in shift_test.CONDITIONS if c[0] in TEST_CONDS]
for speaker, pinyin in TEST_CLIPS:
    rec = find(TEST_M, speaker, pinyin)
    if not rec: continue
    print(f"\n===== TEST  {speaker} {pinyin}")
    _, versions = shift_test.make_versions((rec, conds))
    for name in TEST_CONDS:
        if versions.get(name) is not None:
            play(versions[name], pinyin, name)

# ---------------------------------------------------------- 2. what the model was TRAINED on
def praat_fixed(wav, formant, semitones, tempo=1.0):
    """data.praat_voice with chosen settings instead of random ones."""
    import parselmouth
    from parselmouth.praat import call
    snd = parselmouth.Sound(wav.astype(np.float64), sampling_frequency=SAMPLE_RATE)
    median = call(snd.to_pitch(pitch_floor=75.0, pitch_ceiling=600.0), "Get quantile", 0, 0, 0.5, "Hertz")
    if median is None or not np.isfinite(median):
        floor, ceiling, new = 75.0, 600.0, 0.0
    else:
        floor, ceiling, new = max(60.0, median / 2), min(800.0, median * 2), median * 2 ** (semitones / 12)
    out = call(snd, "Change gender", floor, ceiling, float(formant), float(new), 1.0, 1.0 / tempo)
    return pad_audio(normalize_loudness(np.asarray(out.values[0], dtype=np.float32)))

TRAIN_CLIPS = [("hotoo", "feng1"), ("yabla", "feng1"), ("MV1", "fang1"), ("FV1", "feng1")]
EXTREMES = [(1.00, 0, "unchanged"), (1.13, 0, "resonances x1.13 (top of range)"),
            (0.88, 0, "resonances x0.88 (bottom)"), (1.00, -4, "pitch -4 st"),
            (1.13, -4, "resonances x1.13 + pitch -4"), (0.88, +4, "resonances x0.88 + pitch +4")]
rng = np.random.default_rng(0)
for speaker, pinyin in TRAIN_CLIPS:
    rec = find(TRAIN_M, speaker, pinyin)
    if not rec: continue
    print(f"\n===== TRAIN  {speaker} {pinyin}")
    wav = load_audio(rec["audio"])
    for f, st, label in EXTREMES:
        play(praat_fixed(wav, f, st), pinyin, label)
    for i in range(2):                                   # two genuinely random draws, as in training
        play(data.praat_voice(wav, rng), pinyin, f"random draw {i + 1}")

In [ ]:
%pip install 

In [ ]:
%run -i debug_rescorer.py

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

PAIR = ("eng", "ang")      # or ("en", "an")
POINT = 30                 # % of the voiced region to plot (try 30 too: the nasal is in the voiced region)
COL = {PAIR[0]: "#2a78d6", PAIR[1]: "#eb6834"}

x = d[d.final.isin(PAIR)]
voices = sorted(x.voice.unique(), key=lambda v: (v in ("FV3", "MV3", "studycli", "uvm"), v))
nc = 4
fig, axes = plt.subplots(math.ceil(len(voices) / nc), nc, figsize=(16, 4 * math.ceil(len(voices) / nc)))
for ax, v in zip(axes.flat, voices):
    y = x[x.voice == v]
    for fin in PAIR:
        z = y[(y.final == fin) & ~y.misheard]
        ax.scatter(z[f"F2_{POINT}"], z[f"F1_{POINT}"], s=22, color=COL[fin], alpha=0.7, label=fin)
    z = y[y.misheard & (y.final == PAIR[0])]
    if len(z):
        ax.scatter(z[f"F2_{POINT}"], z[f"F1_{POINT}"], s=60, marker="x", color="#111",
                   label=f"{PAIR[0]} heard as {PAIR[1]}")
    ax.invert_xaxis(); ax.invert_yaxis()          # phonetics convention: open vowels low, front vowels left
    ax.set_title(f"{v}  (f0 {y.f0.median():.0f} Hz, ceiling {y.ceiling.median():.0f})", fontsize=10)
    ax.set_xlabel("F2 (Hz)"); ax.set_ylabel("F1 (Hz)")
    ax.grid(color="#ddd", lw=0.6)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
for ax in axes.flat[len(voices):]:
    ax.axis("off")
h, l = [], []
for ax in axes.flat[:len(voices)]:
    for hh, ll in zip(*ax.get_legend_handles_labels()):
        if ll not in l:
            h.append(hh); l.append(ll)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.suptitle(f"F1 vs F2 at {POINT}% of the voiced region", y=0.995)
fig.legend(h, l, loc="upper center", bbox_to_anchor=(0.5, 0.975), ncol=3, frameon=False)
plt.show()

# within-speaker: train and test on one voice's own clips (5-fold cross-validation)
FEATS = [f"F{i}_{p}" for i in (1, 2, 3) for p in (30, 50, 70)]
clf = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))
rows = []
for v in voices:
    y = x[x.voice == v]
    lab = (y.final == PAIR[0]).values
    if min(lab.sum(), (~lab).sum()) < 5:
        continue
    pred = cross_val_predict(clf, y[FEATS], lab, cv=StratifiedKFold(5, shuffle=True, random_state=0))
    mis = (y.misheard & (y.final == PAIR[0])).values
    rows.append({"voice": v, "clips": len(y), "within-voice accuracy": f"{(pred == lab).mean():.0%}",
                 f"misheard {PAIR[0]} put right": f"{(pred[mis] == lab[mis]).sum()}/{mis.sum()}" if mis.any() else "-"})
print(f"Formants only, {PAIR[0]} vs {PAIR[1]}, trained and tested within each voice (chance = 50%)")
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
P = list(range(10, 100, 10))
GROUPS = [("eng", False, "eng, heard right", "#2a78d6", "-"),
          ("eng", True,  "eng, heard as ang", "#111", "--"),
          ("ang", False, "ang", "#eb6834", "-")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for ax, v in zip(axes, ["uvm", "studycli"]):
    for fin, mis, label, c, ls in GROUPS:
        g = d[(d.voice == v) & (d.final == fin) & (d.misheard == mis)]
        if len(g):
            ax.plot(P, g[[f"F1_{p}" for p in P]].median(), color=c, ls=ls, lw=2, marker="o", ms=4,
                    label=f"{label} (n={len(g)})")
    ax.set_title(v); ax.set_xlabel("% of voiced region"); ax.set_ylabel("median F1 (Hz)")
    ax.grid(color="#ddd", lw=0.6); ax.legend(frameon=False, fontsize=8)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================== CELL 5: judge the vowel against the clip's own nasal
# Uses `d` from cell 1. eng/ang (and en/an) end in the same nasal, so the nasal's resonances are a
# yardstick for the speaker's size that doesn't depend on which vowel was said, inside every clip.
# No F3, no calibration, no history: one clip at a time.
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

HELD_OUT = ["FV3", "MV3", "studycli", "uvm"]
bark = lambda f: 26.81 * f / (1960 + f) - 0.53

z = d.copy()
for i in (1, 2):
    vowel = z[[f"F{i}_{p}" for p in (20, 30, 40)]].median(axis=1)    # early vowel
    nasal = z[[f"F{i}_{p}" for p in (80, 90)]].median(axis=1)        # the nasal ending
    z[f"vF{i}"], z[f"nF{i}"] = vowel, nasal
    z[f"rel{i}"] = bark(vowel) - bark(nasal)                          # vowel measured against the nasal
z["v21"] = bark(z.vF2) - bark(z.vF1)                                   # F2-F1 within the vowel

SETS = {"vowel only, raw Hz": ["vF1", "vF2"],
        "vowel vs its own nasal": ["rel1", "rel2", "v21"]}

def clf():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))

print("Formants only, one clip at a time. Trained on the 8 training voices, tested on the 4 held-out (chance = 50%)\n")
for a, b in [("eng", "ang"), ("en", "an")]:
    tr = z[~z.voice.isin(HELD_OUT) & z.final.isin([a, b])]
    rows = []
    for name, feats in SETS.items():
        m = clf().fit(tr[feats], tr.final == a)
        for v in HELD_OUT:
            te = z[(z.voice == v) & z.final.isin([a, b])]
            pred, lab = m.predict(te[feats]), (te.final == a).values
            mis = (te.misheard & (te.final == a)).values
            rows.append({"features": name, "voice": v, "accuracy": f"{(pred == lab).mean():.0%}",
                         "model's errors put right": f"{(pred[mis] == lab[mis]).sum()}/{mis.sum()}"})
    print(f"=== {a} vs {b}")
    print(pd.DataFrame(rows).pivot(index="voice", columns="features").to_string(), "\n")

# is the nasal really vowel-independent? Per voice, its F1/F2 should be about the same after eng and ang
print("Median nasal resonances per voice (should be similar for both finals; differ between voices):")
print(z[z.final.isin(["eng", "ang"])].groupby(["voice", "final"])[["nF1", "nF2"]].median().round(0)
      .unstack("final").to_string())

In [ ]:
VOICE = "uvm"
FIN, PARTNER = "eng", "ang"          # or "en", "an"
mis  = d[(d.voice == VOICE) & (d.final == FIN) & d.misheard].index[:4]
good = d[(d.voice == VOICE) & (d.final == FIN) & ~d.misheard].index[:2]
part = d[(d.voice == VOICE) & (d.final == PARTNER)].index[:2]
for a in list(mis) + list(good) + list(part):
    inspect(a)

In [ ]:
# ============================================================== CELL 1: raw formant tracks (≈3 min, saved to Drive)
# Re-measures F1/F2/F3 in Hz at 10%, 20%, ... 90% of the voiced region, with the same settings as
# before, so the earlier features can be checked against raw values.
import json, math, os, warnings
import numpy as np, pandas as pd
from multiprocessing import Pool
from pinyin_units import split_syllable

runs = "/content/drive/MyDrive/pinyin_runs"
MANIFESTS = ["manifests/syl_train.jsonl", "manifests/good/train.jsonl",
             "manifests/syl_test.jsonl", "manifests/good/test_studycli.jsonl", "manifests/good/test_uvm.jsonl"]
FINALS = ("eng", "ang", "en", "an")
RAW_CSV = f"{runs}/formant_raw.csv"
FRACS = [i / 10 for i in range(1, 10)]

def raw_track(audio):
    import parselmouth
    from data import load_audio
    out = {"audio": audio}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            snd = parselmouth.Sound(load_audio(audio).astype(np.float64), sampling_frequency=16000)
            pitch = snd.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=600)
            f0, times = pitch.selected_array["frequency"], pitch.xs()
            voiced = times[f0 > 0]
            if len(voiced) < 5:
                return out
            t0, t1 = float(voiced[0]), float(voiced[-1])
            med = float(np.median(f0[f0 > 0]))
            ceiling = 5500 if med > 165 else 5000
            fmt = snd.to_formant_burg(time_step=0.005, max_number_of_formants=5, maximum_formant=ceiling)
            out.update(t0=t0, t1=t1, f0=med, ceiling=ceiling)
            for frac in FRACS:
                t = t0 + frac * (t1 - t0)
                for i in (1, 2, 3):
                    out[f"F{i}_{int(frac * 100)}"] = np.nanmedian(
                        [fmt.get_value_at_time(i, t + d) for d in (-0.01, 0, 0.01)])
    except Exception:
        pass
    return out

rows = []
for m in MANIFESTS:
    for line in open(m):
        r = json.loads(line)
        p = split_syllable(r["pinyin"])
        if p and p[1] in FINALS:
            rows.append({"audio": r["audio"], "voice": r.get("speaker") or r.get("source"),
                         "pinyin": r["pinyin"], "final": p[1]})
clips = pd.DataFrame(rows).drop_duplicates("audio").set_index("audio")

if os.path.exists(RAW_CSV):
    raw = pd.read_csv(RAW_CSV).set_index("audio")
else:
    with Pool(2) as pool:
        raw = pd.DataFrame(pool.map(raw_track, list(clips.index), chunksize=16)).set_index("audio")
    raw.to_csv(RAW_CSV)
d = clips.join(raw)

# what 6b + 7 heard (held-out voices only), to mark the misheard clips
Ps = []
for name in ["stage6b", "stage7_hubert"]:
    x = pd.read_csv(f"{runs}/{name}_probs.csv", dtype=str, keep_default_na=False).set_index("audio")
    cols = [c for c in x.columns if c.startswith("final:") and "silence" not in c]
    Ps.append(np.exp(x[cols].astype(float)))
common = Ps[0].index.intersection(Ps[1].index)
d["heard"] = ((Ps[0].loc[common] + Ps[1].loc[common]) / 2).idxmax(axis=1).str.split(":").str[1].reindex(d.index)
d["misheard"] = d.heard.notna() & (d.heard != d.final)

# consistency with the features the rescorer used: F1/F3 at 50% should equal its r13_50
old = pd.read_csv(f"{runs}/formant_feats.csv").set_index("audio")["r13_50"]
diff = (d.F1_50 / d.F3_50 - old.reindex(d.index)).abs()
print(f"{len(d)} clips. Re-measured F1/F3 matches the saved r13_50 on {(diff < 1e-6).mean():.0%} "
      f"(should be ~100%: same code, same settings)")
print(d.groupby("voice")[["f0", "ceiling"]].median().round(0).to_string())


In [ ]:
# ============================================================== CELL 6: is uvm's recording quality the cause?
# Uses `d` from cell 1, plus shift_test.py and score.py from the repo. About 5 minutes on the GPU.
#  Part A: average spectrum of each voice. Shows whether uvm is missing low frequencies or has a noise floor.
#  Part B: make CLEAN voices sound like uvm's channel, then see if the 6b+7 model starts hearing eng as ang.
#          No training: if the clean voices break the same way, the channel is the cause.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import welch, butter, sosfiltfilt, firwin2
from data import load_audio
from score import Scorer
import shift_test

runs = "/content/drive/MyDrive/pinyin_runs"
scorers = [Scorer(f"{runs}/stage2/best")]
SR = 16000
SHOW = ["uvm", "studycli", "FV3", "MV3", "hotoo", "yabla"]
TEST = ["FV3", "MV3", "studycli"]               # clean voices to push through uvm's channel
clips = d[d.final.isin(["eng", "ang"])]

# ---------------------------------------------------------------- A. average spectrum per voice
spec = {}
for v in SHOW:
    P = []
    for a in clips[clips.voice == v].index[:80]:
        f, p = welch(load_audio(a), fs=SR, nperseg=512)
        P.append(p)
    spec[v] = np.mean(P, axis=0)
band = (f >= 300) & (f <= 3000)
COL = ["#111", "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#6250d6"]
fig, ax = plt.subplots(figsize=(11, 4))
for (v, p), c in zip(spec.items(), COL):
    db = 10 * np.log10(p)
    ax.plot(f, db - db[band].mean(), color=c, lw=2.5 if v == "uvm" else 1.5, label=v)
ax.set_xscale("log"); ax.set_xlim(60, 8000)
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("dB, relative to 300-3000 Hz average")
ax.grid(color="#ddd", lw=0.6); ax.legend(frameon=False, ncol=6)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.set_title("Average spectrum per voice (eng/ang clips, silence included)")
plt.show()
low = (f >= 100) & (f < 300)
print("energy at 100-300 Hz (where the nasal hum and low F1 live), relative to 300-3000 Hz:")
for v, p in spec.items():
    print(f"  {v:9} {10 * np.log10(p[low].mean() / p[band].mean()):6.1f} dB")

# ---------------------------------------------------------------- B. give clean voices uvm's channel
def smooth(x, k=5):
    return np.convolve(x, np.ones(k) / k, mode="same")

EQ = {}
for v in TEST:   # filter that turns voice v's average spectrum into uvm's
    g = smooth(np.sqrt(spec["uvm"] / spec[v]))
    g = np.clip(g / np.median(g[band]), 0.03, 10)
    EQ[v] = firwin2(513, f, g, fs=SR)
HP = butter(4, 300, "highpass", fs=SR, output="sos")
CONDS = {
    "as recorded": lambda w, v: w,
    "low frequencies cut (300 Hz)": lambda w, v: sosfiltfilt(HP, w),
    "uvm's spectrum": lambda w, v: np.convolve(w, EQ[v], mode="same"),
}

scorers = [Scorer(f"{runs}/stage2/best")]
def heard(wav, pinyin):
    ps = [shift_test.get_probs(s, wav.astype(np.float32), pinyin)["final"] for s in scorers]
    return shift_test.top_unit({k: np.mean([q[k] for q in ps]) for k in ps[0]})

rows = []
todo = clips[clips.voice.isin(TEST)]
for i, (a, r) in enumerate(todo.iterrows(), 1):
    wav = load_audio(a)
    row = {"voice": r.voice, "final": r.final}
    for name, fn in CONDS.items():
        row[name] = heard(fn(wav, r.voice), r.pinyin)
    rows.append(row)
    if i % 100 == 0:
        print(f"  {i}/{len(todo)} clips", flush=True)
res = pd.DataFrame(rows)

out = []
for (v, fin), g in res.groupby(["voice", "final"]):
    partner = "ang" if fin == "eng" else "eng"
    r = {"voice": v, "clips said": fin, "partner": partner, "n": len(g)}
    for name in CONDS:
        r[name] = (g[name] == partner).sum()
    out.append(r)
print("\nClips the model hears as the PARTNER vowel, as recorded vs through uvm-like channels:")
print(pd.DataFrame(out).to_string(index=False))

In [ ]:
# ============================================================== CELL 7: does the nasal ending mislead the vowel decision?
# Uses `d` from cell 1 (needs its t0/t1 columns) and the two saved models. No training, ~3 minutes.
# Each eng/ang clip is cut off before the nasal and scored again. "feng" becomes "fe", "fang" becomes "fa".
# If uvm's misheard eng clips come out e-like once the nasal is gone, the vowel was fine and the ending
# was what pushed the model to "ang". That's the case where a vowel-only head would help.
import numpy as np, pandas as pd
from data import load_audio
from score import Scorer
import shift_test

runs = "/content/drive/MyDrive/pinyin_runs"
SR = 16000
CUT = 0.55            # keep the clip up to 55% of the voiced region (the nasal starts around 60-70%)
VOICES = ["uvm", "studycli", "FV3", "MV3"]
E_LIKE = {"e", "en", "eng", "ei", "er"}
A_LIKE = {"a", "an", "ang", "ao", "ai"}

scorers = [Scorer(f"{runs}/stage2/best")]
def heard(wav, pinyin):
    ps = [shift_test.get_probs(s, wav.astype(np.float32), pinyin)["final"] for s in scorers]
    return shift_test.top_unit({k: np.mean([q[k] for q in ps]) for k in ps[0]})

def vowel_only(wav, t0, t1):
    end = int((t0 + CUT * (t1 - t0)) * SR)
    w = wav[:end].copy()
    fade = min(160, len(w))                       # 10 ms fade so the cut doesn't click
    w[-fade:] *= np.linspace(1, 0, fade)
    return np.concatenate([w, np.zeros(int(0.15 * SR), np.float32)])

todo = d[d.voice.isin(VOICES) & d.final.isin(["eng", "ang"]) & d.t0.notna()]
rows = []
for i, (a, r) in enumerate(todo.iterrows(), 1):
    wav = load_audio(a)
    cut = heard(vowel_only(wav, r.t0, r.t1), r.pinyin)
    rows.append({"voice": r.voice, "final": r.final, "full clip": heard(wav, r.pinyin), "vowel only": cut,
                 "group": "e" if cut in E_LIKE else "a" if cut in A_LIKE else "other"})
    if i % 100 == 0:
        print(f"  {i}/{len(todo)} clips", flush=True)
res = pd.DataFrame(rows)

out = []
for v in VOICES:
    for fin, want in [("eng", "e"), ("ang", "a")]:
        for label, sel in [("heard right", lambda g: g["full clip"] == fin),
                           ("heard as partner", lambda g: g["full clip"] != fin)]:
            g = res[(res.voice == v) & (res.final == fin)]
            g = g[sel(g)]
            if len(g):
                out.append({"voice": v, "clips said": fin, "full clip": label, "n": len(g),
                            "vowel only: e-like": (g.group == "e").sum(),
                            "a-like": (g.group == "a").sum(), "other": (g.group == "other").sum()})
print(f"Clips cut at {CUT:.0%} of the voiced region, scored by 6b+7.")
print("Key rows: 'eng, heard as partner'. Mostly e-like => the nasal ending was what misled the model.\n")
print(pd.DataFrame(out).to_string(index=False))
print("\nMost common answers on the cut clips:",
      res.groupby("final")["vowel only"].agg(lambda s: s.value_counts().head(4).to_dict()).to_dict())

In [ ]:
# ============================================================== CELL 1: raw formant tracks (≈3 min, saved to Drive)
# Re-measures F1/F2/F3 in Hz at 10%, 20%, ... 90% of the voiced region, with the same settings as
# before, so the earlier features can be checked against raw values.
import json, math, os, warnings
import numpy as np, pandas as pd
from multiprocessing import Pool
from pinyin_units import split_syllable

runs = "/content/drive/MyDrive/pinyin_runs"
MANIFESTS = ["manifests/syl_train.jsonl", "manifests/good/train.jsonl",
             "manifests/syl_test.jsonl", "manifests/good/test_studycli.jsonl", "manifests/good/test_uvm.jsonl"]
FINALS = ("eng", "ang", "en", "an")
RAW_CSV = f"{runs}/formant_raw.csv"
FRACS = [i / 10 for i in range(1, 10)]

def raw_track(audio):
    import parselmouth
    from data import load_audio
    out = {"audio": audio}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            snd = parselmouth.Sound(load_audio(audio).astype(np.float64), sampling_frequency=16000)
            pitch = snd.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=600)
            f0, times = pitch.selected_array["frequency"], pitch.xs()
            voiced = times[f0 > 0]
            if len(voiced) < 5:
                return out
            t0, t1 = float(voiced[0]), float(voiced[-1])
            med = float(np.median(f0[f0 > 0]))
            ceiling = 5500 if med > 165 else 5000
            fmt = snd.to_formant_burg(time_step=0.005, max_number_of_formants=5, maximum_formant=ceiling)
            out.update(t0=t0, t1=t1, f0=med, ceiling=ceiling)
            for frac in FRACS:
                t = t0 + frac * (t1 - t0)
                for i in (1, 2, 3):
                    out[f"F{i}_{int(frac * 100)}"] = np.nanmedian(
                        [fmt.get_value_at_time(i, t + d) for d in (-0.01, 0, 0.01)])
    except Exception:
        pass
    return out

rows = []
for m in MANIFESTS:
    for line in open(m):
        r = json.loads(line)
        p = split_syllable(r["pinyin"])
        if p and p[1] in FINALS:
            rows.append({"audio": r["audio"], "voice": r.get("speaker") or r.get("source"),
                         "pinyin": r["pinyin"], "final": p[1]})
clips = pd.DataFrame(rows).drop_duplicates("audio").set_index("audio")

if os.path.exists(RAW_CSV):
    raw = pd.read_csv(RAW_CSV).set_index("audio")
else:
    with Pool(2) as pool:
        raw = pd.DataFrame(pool.map(raw_track, list(clips.index), chunksize=16)).set_index("audio")
    raw.to_csv(RAW_CSV)
d = clips.join(raw)

# what 6b + 7 heard (held-out voices only), to mark the misheard clips
Ps = []
for name in ["stage6b", "stage7_hubert"]:
    x = pd.read_csv(f"{runs}/{name}_probs.csv", dtype=str, keep_default_na=False).set_index("audio")
    cols = [c for c in x.columns if c.startswith("final:") and "silence" not in c]
    Ps.append(np.exp(x[cols].astype(float)))
common = Ps[0].index.intersection(Ps[1].index)
d["heard"] = ((Ps[0].loc[common] + Ps[1].loc[common]) / 2).idxmax(axis=1).str.split(":").str[1].reindex(d.index)
d["misheard"] = d.heard.notna() & (d.heard != d.final)

# consistency with the features the rescorer used: F1/F3 at 50% should equal its r13_50
old = pd.read_csv(f"{runs}/formant_feats.csv").set_index("audio")["r13_50"]
diff = (d.F1_50 / d.F3_50 - old.reindex(d.index)).abs()
print(f"{len(d)} clips. Re-measured F1/F3 matches the saved r13_50 on {(diff < 1e-6).mean():.0%} "
      f"(should be ~100%: same code, same settings)")
print(d.groupby("voice")[["f0", "ceiling"]].median().round(0).to_string())


In [ ]:
# ============================================================== CELL 8: listen to the cut clips yourself
# Uses `d` from cell 1. For each clip: spectrogram with the cut point marked, what the model heard
# for the full clip and for the cut clip, then the full audio and the cut audio to play.
import numpy as np, matplotlib.pyplot as plt, parselmouth
from IPython.display import Audio, display
from data import load_audio
from score import Scorer
import shift_test

assert {"final", "voice", "misheard", "t0"} <= set(d.columns), \
    "`d` was overwritten by another cell: run cell 1 again first"

runs = "/content/drive/MyDrive/pinyin_runs"
SR = 16000
MODELS = ["stage2"]        # trained without voice or noise augmentation. ["stage6b", "stage7_hubert"] = the combined model
CUT = 0.55                 # fraction of the voiced region kept; try 0.45 or 0.65 and listen again
PICK = [                   # (voice, final, misheard by 6b+7?, how many)
    ("uvm", "eng", True, 3),
    ("uvm", "eng", False, 2),
    ("uvm", "ang", False, 2),
    ("studycli", "eng", True, 3),
    ("studycli", "ang", False, 1),
]

scorers = [Scorer(f"{runs}/{m}/best") for m in MODELS]
def heard(wav, pinyin):
    ps = [shift_test.get_probs(s, wav.astype(np.float32), pinyin)["final"] for s in scorers]
    p = {k: np.mean([q[k] for q in ps]) for k in ps[0]}
    top = shift_test.top_unit(p)
    return f"{top} ({p[top]:.2f})"

def vowel_only(wav, t0, t1, cut=CUT):
    end = int((t0 + cut * (t1 - t0)) * SR)
    w = wav[:end].copy()
    fade = min(160, len(w))
    w[-fade:] *= np.linspace(1, 0, fade)
    return np.concatenate([w, np.zeros(int(0.15 * SR), np.float32)])

def show(audio):
    r = d.loc[audio]
    wav = load_audio(audio)
    cut = vowel_only(wav, r.t0, r.t1)
    t_cut = r.t0 + CUT * (r.t1 - r.t0)
    spec = parselmouth.Sound(wav.astype(np.float64), sampling_frequency=SR).to_spectrogram(
        window_length=0.005, maximum_frequency=5000)
    db = 10 * np.log10(np.maximum(spec.values, 1e-12))
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.pcolormesh(spec.xs(), spec.ys(), db, cmap="Greys", vmin=db.max() - 70, shading="auto")
    ax.axvspan(t_cut, spec.xs()[-1], color="#e34948", alpha=0.15, label="removed")
    ax.axvline(t_cut, color="#e34948", lw=1.5)
    ax.set_ylim(0, 5000); ax.set_xlabel("time (s)"); ax.set_ylabel("Hz")
    ax.set_title(f"{r.voice} {r.pinyin}   full clip heard {heard(wav, r.pinyin)}   "
                 f"cut clip heard {heard(cut, r.pinyin)}", fontsize=10)
    ax.legend(loc="upper right", frameon=False, fontsize=8)
    plt.show()
    print("full:"); display(Audio(wav, rate=SR))
    print("cut:");  display(Audio(cut, rate=SR))

print(f"model: {' + '.join(MODELS)}   cut at {CUT:.0%} of the voiced region")
for voice, fin, mis, n in PICK:
    sel = d[(d.voice == voice) & (d.final == fin) & (d.misheard == mis) & d.t0.notna()].index[:n]
    print(f"\n==================== {voice} {fin}, {'misheard' if mis else 'heard right'} by 6b+7")
    for a in sel:
        show(a)

In [ ]:
# ============================================================== CELL 9: measure F3 properly, then test F1/F3 across speakers
# Uses `d` from cell 1 (for each clip's voiced region and what 6b+7 heard). About 5 minutes, CPU only.
# For every eng/ang/en/an clip, Praat measures the vowel (20-50% of the voiced region, before the nasal)
# at 9 formant ceilings from 4500 to 6500 Hz, and keeps the ceiling whose F1-F3 tracks are steadiest.
# This works one clip at a time: no calibration, no other clips from the speaker.
import math, os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from multiprocessing import Pool
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

assert {"final", "voice", "misheard", "t0"} <= set(d.columns), \
    "`d` was overwritten by another cell: run cell 1 again first"

runs = "/content/drive/MyDrive/pinyin_runs"
CEIL_CSV = f"{runs}/formant_ceiling.csv"       # cached, so reruns skip the measuring
CEILINGS = list(range(4500, 6501, 250))
WIN = (0.20, 0.50)                             # part of the voiced region treated as the vowel
HELD_OUT = ["FV3", "MV3", "studycli", "uvm"]

def measure(args):
    audio, t0, t1 = args
    import parselmouth
    from data import load_audio
    out = {"audio": audio}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            snd = parselmouth.Sound(load_audio(audio).astype(np.float64), sampling_frequency=16000)
            ta, tb = t0 + WIN[0] * (t1 - t0), t0 + WIN[1] * (t1 - t0)
            part = snd.extract_part(from_time=max(0.0, ta - 0.05), to_time=min(snd.duration, tb + 0.05),
                                    preserve_times=True)
            ts = np.arange(ta, tb, 0.005)
            if len(ts) < 4:
                return out
            best = None
            for c in CEILINGS:
                fmt = part.to_formant_burg(time_step=0.005, max_number_of_formants=5, maximum_formant=c)
                F = np.array([[fmt.get_value_at_time(i, t) for t in ts] for i in (1, 2, 3)])
                ok = np.isfinite(F).all(0) & (F[0] < F[1]) & (F[1] < F[2])
                if ok.mean() < 0.6:
                    continue
                L = np.log(F[:, ok])
                jump = np.abs(np.diff(L, axis=1)).mean() if L.shape[1] > 1 else 1.0   # track flips
                score = 100 * jump + 10 * L.std(axis=1).sum() + 5 * (1 - ok.mean())
                if best is None or score < best[0]:
                    best = (score, c, np.median(F[:, ok], axis=1), L.std(axis=1))
            if best is not None:
                score, c, med, sd = best
                out.update(ceiling=c, score=score, F1=med[0], F2=med[1], F3=med[2], F3_sd=float(sd[2]))
    except Exception:
        pass
    return out

todo = d[d.t0.notna()]
if os.path.exists(CEIL_CSV):
    meas = pd.read_csv(CEIL_CSV).set_index("audio")
else:
    with Pool(2) as pool:
        meas = pd.DataFrame(pool.map(measure, list(zip(todo.index, todo.t0, todo.t1)), chunksize=16))
    meas = meas.set_index("audio")
    meas.to_csv(CEIL_CSV)

z = d[["voice", "final", "pinyin", "misheard"]].join(meas, how="inner")
print(f"measured {z.F3.notna().mean():.0%} of {len(z)} clips")

# ---------------------------------------------------------------- 1. is the new F3 steadier than the old one?
old_sd = np.log(d[["F3_20", "F3_30", "F3_40", "F3_50"]]).std(axis=1)
z["old_F3_sd"] = old_sd.reindex(z.index)
print("\nF3 steadiness within the vowel (spread of log F3; lower = steadier) and chosen ceilings:")
print(z.groupby("voice").agg(old=("old_F3_sd", "median"), new=("F3_sd", "median"),
                             F3_Hz=("F3", "median"), ceiling=("ceiling", "median")).round(3).to_string())

# ---------------------------------------------------------------- 2. one boundary for every speaker?
bark = lambda f: 26.81 * f / (1960 + f) - 0.53
z["F1/F3"], z["F2/F3"] = z.F1 / z.F3, z.F2 / z.F3
z["b21"], z["b32"] = bark(z.F2) - bark(z.F1), bark(z.F3) - bark(z.F2)
SETS = {"F1, F2 in Hz": ["F1", "F2"],
        "F1/F3, F2/F3": ["F1/F3", "F2/F3"],
        "ratios + Bark gaps": ["F1/F3", "F2/F3", "b21", "b32"]}

def clf():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))

print("\nOne clip at a time. Trained on the 8 training voices, tested on the 4 held-out (chance = 50%)")
for a, b in [("eng", "ang"), ("en", "an")]:
    tr = z[~z.voice.isin(HELD_OUT) & z.final.isin([a, b])]
    rows = []
    for name, feats in SETS.items():
        m = clf().fit(tr[feats], tr.final == a)
        for v in HELD_OUT:
            te = z[(z.voice == v) & z.final.isin([a, b])]
            pred, lab = m.predict(te[feats]), (te.final == a).values
            mis = (te.misheard & (te.final == a)).values
            rows.append({"features": name, "voice": v, "accuracy": f"{(pred == lab).mean():.0%}",
                         "model's errors put right": f"{(pred[mis] == lab[mis]).sum()}/{mis.sum()}"})
    print(f"\n=== {a} vs {b}")
    print(pd.DataFrame(rows).pivot(index="voice", columns="features").to_string())

# picture: F1/F3 for every voice. If one horizontal line splits eng from ang for all voices, it works.
COL = {"eng": "#2a78d6", "ang": "#eb6834"}
x = z[z.final.isin(["eng", "ang"])]
voices = sorted(x.voice.unique(), key=lambda v: (v in HELD_OUT, v))
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(13, 4.5))
for k, v in enumerate(voices):
    for j, fin in enumerate(["eng", "ang"]):
        y = x[(x.voice == v) & (x.final == fin)]
        xs = k + (j - 0.5) * 0.35 + rng.uniform(-0.08, 0.08, len(y))
        ax.scatter(xs, y["F1/F3"], s=12, color=COL[fin], alpha=0.6, label=fin if k == 0 else None)
        mm = y.misheard.values
        if mm.any():
            ax.scatter(xs[mm], y["F1/F3"][mm], s=45, marker="x", color="#111",
                       label="eng heard as ang" if "eng heard as ang" not in ax.get_legend_handles_labels()[1] else None)
ax.set_xticks(range(len(voices)))
ax.set_xticklabels([v + ("\n(held out)" if v in HELD_OUT else "") for v in voices], fontsize=9)
ax.set_ylabel("F1 / F3 in the vowel")
ax.grid(axis="y", color="#ddd", lw=0.6)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.legend(frameon=False, ncol=3, loc="upper left")
ax.set_title("F1 relative to F3, one clip at a time: does one line split eng from ang for every voice?")
plt.show()

# ---------------------------------------------------------------- 3. look at the new tracks yourself
from IPython.display import Audio, display
import parselmouth
from data import load_audio

def inspect_new(audio):
    r, q = d.loc[audio], z.loc[audio]
    wav = load_audio(audio)
    snd = parselmouth.Sound(wav.astype(np.float64), sampling_frequency=16000)
    spec = snd.to_spectrogram(window_length=0.005, maximum_frequency=5000)
    fmt = snd.to_formant_burg(time_step=0.005, max_number_of_formants=5, maximum_formant=q.ceiling)
    db = 10 * np.log10(np.maximum(spec.values, 1e-12))
    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.pcolormesh(spec.xs(), spec.ys(), db, cmap="Greys", vmin=db.max() - 70, shading="auto")
    ta, tb = r.t0 + WIN[0] * (r.t1 - r.t0), r.t0 + WIN[1] * (r.t1 - r.t0)
    ts = [t for t in fmt.xs() if r.t0 - 0.03 <= t <= r.t1 + 0.03]
    for i, c in zip((1, 2, 3), ("#e34948", "#1baf7a", "#2a78d6")):
        ax.scatter(ts, [fmt.get_value_at_time(i, t) for t in ts], s=4, color=c, label=f"F{i}")
    ax.axvspan(ta, tb, color="#eda100", alpha=0.15, label="vowel window")
    ax.set_ylim(0, 5000); ax.set_xlabel("time (s)"); ax.set_ylabel("Hz")
    ax.set_title(f"{r.voice} {r.pinyin}  ceiling {q.ceiling:.0f}  F1 {q.F1:.0f}  F2 {q.F2:.0f}  F3 {q.F3:.0f}  "
                 f"F1/F3 {q['F1/F3']:.3f}  (6b+7 heard {r.heard})", fontsize=9)
    ax.legend(loc="upper right", fontsize=7, ncol=4, frameon=False)
    plt.show()
    display(Audio(wav, rate=16000))

for v, n in [("studycli", 3), ("uvm", 3)]:
    for a in z[(z.voice == v) & (z.final == "eng") & z.misheard].index[:n]:
        inspect_new(a)
    for a in z[(z.voice == v) & (z.final == "ang")].index[:1]:
        inspect_new(a)

In [ ]:
import numpy as np, pandas as pd
runs = "/content/drive/MyDrive/pinyin_runs"
Ps = []
for name in ["stage6b", "stage7_hubert"]:
    x = pd.read_csv(f"{runs}/{name}_probs.csv", dtype=str, keep_default_na=False).set_index("audio")
    Ps.append(np.exp(x[["final:eng", "final:ang"]].astype(float)))
    meta = x[["source", "target_final"]]
idx = Ps[0].index.intersection(Ps[1].index)
P = (Ps[0].loc[idx] + Ps[1].loc[idx]) / 2
m = meta.loc[idx]
m["eng share"] = P["final:eng"] / (P["final:eng"] + P["final:ang"])

BINS = [0, 0.05, 0.5, 0.95, 1.0001]
LABELS = ["clearly ang (<5%)", "leans ang (5-50%)", "leans eng (50-95%)", "clearly eng (>95%)"]
for fin in ["eng", "ang"]:
    t = m[m.target_final == fin]
    tab = pd.crosstab(t.source, pd.cut(t["eng share"], BINS, labels=LABELS, right=False))
    print(f"\n=== clips said {fin}: how much of the eng/ang probability goes to eng")
    print(tab.to_string())

In [ ]:
# ============================================================== Evaluate with the eng -> ang forgiveness
# Uses the saved probability CSVs (no GPU, a few seconds) and the FORGIVE table from score.py,
# so it measures exactly what the app will do. Needs the updated score.py in the repo folder.
import numpy as np, pandas as pd
from score import FORGIVE, DEFAULT_THRESHOLD as THR

runs = "/content/drive/MyDrive/pinyin_runs"
PARTS = ["initial", "final", "tone"]
WEB = ["studycli", "uvm"]

NNG = {"an": ("ang",), "ang": ("an",), "en": ("eng",), "eng": ("en",), "in": ("ing",), "ing": ("in",),
       "ian": ("iang",), "iang": ("ian",), "uan": ("uang",), "uang": ("uan",)}
RULES = {
    "none": {},
    "eng→ang": {"final": {"eng": ("ang",)}},
    "eng→ang + n/ng": {"final": {**NNG, "eng": ("en", "ang")}},
    "eng→ang + n/ng + en→an": {"final": {**NNG, "eng": ("en", "ang"), "en": ("eng", "an")}},
}

# swapped-target test: a native clip of SAID, scored as if the learner was attempting TARGET
SWAPS = [("final", "ang", "eng"), ("final", "eng", "ang"), ("final", "an", "en"), ("final", "en", "an"),
         ("final", "in", "ing"), ("final", "ing", "in"), ("final", "ou", "ong"), ("final", "ao", "ou"),
         ("initial", "x", "q"), ("initial", "q", "x"), ("initial", "zh", "j"), ("initial", "j", "zh"),
         ("initial", "z", "zh"), ("initial", "s", "sh"), ("tone", "2", "3"), ("tone", "3", "2")]

def load(name):
    x = pd.read_csv(f"{runs}/{name}_probs.csv", dtype=str, keep_default_na=False).set_index("audio")
    cols = [c for c in x.columns if c.split(":")[0] in PARTS]
    return x[["source"] + [f"target_{p}" for p in PARTS]], np.exp(x[cols].astype(float))

meta6, P6 = load("stage6b")
meta7, P7 = load("stage7_hubert")
idx = P6.index.intersection(P7.index)
MODELS = {"6b": (meta6, P6), "7": (meta7, P7),
          "6b+7": (meta6.loc[idx], (P6.loc[idx] + P7.loc[idx]) / 2)}

def p_of(P, part, units, forgive):
    """Probability credited to each clip's target unit, plus any forgiven partners."""
    out = np.zeros(len(P))
    for k, u in enumerate(units):
        out[k] = P.iat[k, P.columns.get_loc(f"{part}:{u}")] + sum(
            P.iat[k, P.columns.get_loc(f"{part}:{f}")] for f in forgive.get(part, {}).get(u, ()))
    return out

def evaluate(meta, P, forgive):
    res = pd.DataFrame({"source": meta.source}, index=meta.index)
    for part in PARTS:
        cols = [c for c in P.columns if c.startswith(part + ":") and "silence" not in c]
        heard = P[cols].idxmax(axis=1).str.split(":").str[1]
        tgt = meta[f"target_{part}"]
        ok_alt = [h in forgive.get(part, {}).get(t, ()) for h, t in zip(heard, tgt)]
        res[f"ok_{part}"] = (heard == tgt) | np.array(ok_alt)
        res[f"flag_{part}"] = p_of(P, part, list(tgt), forgive) < THR
    res["all3"] = res[[f"ok_{p}" for p in PARTS]].all(axis=1)
    res["false_alarm"] = res[[f"flag_{p}" for p in PARTS]].any(axis=1)
    return res

# ---------------------------------------------------------------- 1. accuracy and false alarms
rows = []
for mname, (meta, P) in MODELS.items():
    for rname, rule in RULES.items():
        r = evaluate(meta, P, rule)
        for v, g in list(r.groupby("source")) + [("BOTH WEB VOICES", r[r.source.isin(WEB)])]:
            rows.append({"model": mname, "rule": rname, "voice": v,
                         "all-3 accuracy %": round(100 * g.all3.mean(), 1),
                         "final accuracy %": round(100 * g.ok_final.mean(), 1),
                         "false alarms %": round(100 * g.false_alarm.mean(), 1)})
t = pd.DataFrame(rows)
for mname in MODELS:
    print(f"\n=== {mname}  (threshold {THR})")
    print(t[t.model == mname].pivot(index="voice", columns="rule",
          values=["all-3 accuracy %", "final accuracy %", "false alarms %"]).to_string())

# ---------------------------------------------------------------- 2. the cost: mistakes that now slip through
meta, P = MODELS["6b+7"]
rows = []
for part, said, target in SWAPS:
    m = (meta[f"target_{part}"] == said).values
    if not m.any() or f"{part}:{target}" not in P.columns:
        continue
    r = {"part": part, "clip says": said, "scored as attempt at": target, "clips": int(m.sum())}
    for rname, rule in RULES.items():
        passed = p_of(P[m], part, [target] * m.sum(), rule) >= THR
        r[rname] = f"{100 * passed.mean():.1f}%"
    rows.append(r)
print("\n=== 6b+7: real mistakes that would be marked correct (native clip scored against the wrong target)")
print("    lower is better; 'ang scored as attempt at eng' is the one the new rule gives up on")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
import importlib, score
importlib.reload(score)
from score import FORGIVE, DEFAULT_THRESHOLD as THR

In [ ]:
# ============================================================== CELL 10: tone 2 vs 3 from the pitch contour
# Standalone (doesn't need `d`). Needs the session setup (audio unzipped), praat-parselmouth, and the
# updated score.py in the repo. About 5 minutes, CPU only; pitch tracks are cached on Drive.
#  1. Pitch contour of every tone 2 and tone 3 clip, all 12 voices, in semitones, one clip at a time.
#  2. Picture: average contour per voice, tone 2 vs tone 3, plus the tone 2 clips the model hears as 3.
#  3. A pitch-only tone 2/3 classifier, trained on the 8 TRAINING voices only, tested on the 4 held out.
#  4. The real test: let pitch re-decide the model's 2-vs-3 split, and compare accuracy, false alarms
#     and missed mistakes with the model alone (with the score.py forgiveness rules applied).
import importlib, json, math, os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from multiprocessing import Pool
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from pinyin_units import split_syllable
import score; importlib.reload(score)
from score import FORGIVE, DEFAULT_THRESHOLD as THR

runs = "/content/drive/MyDrive/pinyin_runs"
MANIFESTS = ["manifests/syl_train.jsonl", "manifests/good/train.jsonl",
             "manifests/syl_test.jsonl", "manifests/good/test_studycli.jsonl", "manifests/good/test_uvm.jsonl"]
PITCH_CSV = f"{runs}/pitch_tone23.csv"
HELD_OUT = ["FV3", "MV3", "studycli", "uvm"]
N = 20                                            # contour resampled to 20 points over the voiced part

# ---------------------------------------------------------------- 1. pitch contours
def contour(audio):
    import parselmouth
    from scipy.signal import medfilt
    from data import load_audio
    out = {"audio": audio}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            snd = parselmouth.Sound(load_audio(audio).astype(np.float64), sampling_frequency=16000)
            f = snd.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=600).selected_array["frequency"]
            if (f > 0).sum() < 8:
                return out
            med = float(np.median(f[f > 0]))
            # second pass in a range around this clip's own pitch: avoids most octave jumps
            p = snd.to_pitch(time_step=0.01, pitch_floor=max(60, med / 2.2), pitch_ceiling=min(800, med * 2.2))
            f, t = p.selected_array["frequency"], p.xs()
            v = np.where(f > 0)[0]
            if len(v) < 8:
                return out
            a, b = v[0], v[-1]
            seg = f[a:b + 1].astype(float)
            gaps = seg <= 0                            # unvoiced frames inside the syllable (creak, often tone 3)
            st = np.where(gaps, np.nan, 12 * np.log2(np.maximum(seg, 1) / med))
            idx = np.arange(len(st))
            st = np.interp(idx, idx[~gaps], st[~gaps])
            sm = medfilt(st, 7)
            bad = np.abs(st - sm) > 4                  # leftover jumps: treat as tracking errors
            st = np.interp(idx, idx[~bad], st[~bad]) if (~bad).sum() >= 4 else sm
            st = medfilt(st, 5)
            c = np.interp(np.linspace(0, len(st) - 1, N), idx, st)
            c = c - c[1:4].mean()                      # semitones relative to the start of the syllable
            mid = gaps[int(0.3 * len(gaps)):int(0.7 * len(gaps)) + 1]
            out.update({f"c{i}": c[i] for i in range(N)})
            out.update(dur=float(t[b] - t[a]), f0_med=med, creak=float(mid.mean()) if len(mid) else 0.0)
    except Exception:
        pass
    return out

rows = []
for m in MANIFESTS:
    for line in open(m):
        r = json.loads(line)
        p = split_syllable(r["pinyin"])
        if p and str(p[2]) in ("2", "3"):
            rows.append({"audio": r["audio"], "voice": r.get("speaker") or r.get("source"),
                         "pinyin": r["pinyin"], "tone": str(p[2])})
clips = pd.DataFrame(rows).drop_duplicates("audio").set_index("audio")

if os.path.exists(PITCH_CSV):
    pc = pd.read_csv(PITCH_CSV).set_index("audio")
else:
    with Pool(2) as pool:
        pc = pd.DataFrame(pool.map(contour, list(clips.index), chunksize=32)).set_index("audio")
    pc.to_csv(PITCH_CSV)
t = clips.join(pc, how="inner")
t = t[t.c0.notna()]
C = [f"c{i}" for i in range(N)]
print(f"pitch tracked on {len(t)} of {len(clips)} tone 2/3 clips")

# shape measures, all relative to the syllable's own start (no speaker calibration)
cc = t[C].to_numpy()
inner = cc[:, 2:-2]
t["dip"] = -inner.min(axis=1)                         # how far it falls below the start
t["dip_pos"] = (inner.argmin(axis=1) + 2) / (N - 1)   # where the lowest point is (0 = start, 1 = end)
t["rise"] = cc[:, -3:].mean(axis=1) - inner.min(axis=1)
t["net"] = cc[:, -3:].mean(axis=1)                    # end relative to start
t["first_half"] = cc[:, N // 2]                       # where it is halfway through
SHAPE = ["dip", "dip_pos", "rise", "net", "first_half", "creak"]

# what 6b + 7 heard (held-out voices)
Ps, meta = [], None
for name in ["stage6b", "stage7_hubert"]:
    x = pd.read_csv(f"{runs}/{name}_probs.csv", dtype=str, keep_default_na=False).set_index("audio")
    cols = [c for c in x.columns if c.split(":")[0] in ("initial", "final", "tone")]
    Ps.append(np.exp(x[cols].astype(float)))
    meta = x[["source", "target_initial", "target_final", "target_tone"]] if meta is None else meta
pidx = Ps[0].index.intersection(Ps[1].index)
P = (Ps[0].loc[pidx] + Ps[1].loc[pidx]) / 2
meta = meta.loc[pidx]
tone_cols = [c for c in P.columns if c.startswith("tone:") and "silence" not in c]
heard_tone = P[tone_cols].idxmax(axis=1).str.split(":").str[1]
t["heard"] = heard_tone.reindex(t.index)
print("\nmedian shape per voice (semitones, relative to the syllable start):")
print(t.groupby(["voice", "tone"])[["dip", "dip_pos", "rise", "net"]].median().round(1).unstack("tone").to_string())

# ---------------------------------------------------------------- 2. picture
voices = sorted(t.voice.unique(), key=lambda v: (v in HELD_OUT, v))
x = np.linspace(0, 100, N)
fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharex=True)
for ax, v in zip(axes.flat, voices):
    y = t[t.voice == v]
    for tone, col in [("2", "#2a78d6"), ("3", "#eb6834")]:
        g = y[y.tone == tone]
        ax.plot(x, g[C].median(), color=col, lw=2, label=f"tone {tone} (n={len(g)})")
        ax.fill_between(x, g[C].quantile(0.25), g[C].quantile(0.75), color=col, alpha=0.12)
    g = y[(y.tone == "2") & (y.heard == "3")]
    if len(g):
        ax.plot(x, g[C].median(), color="#111", lw=2, ls="--", label=f"tone 2 heard as 3 (n={len(g)})")
    ax.axhline(0, color="#999", lw=0.8)
    ax.set_title(v + ("  (held out)" if v in HELD_OUT else ""), fontsize=10)
    ax.grid(color="#ddd", lw=0.6)
    ax.legend(frameon=False, fontsize=7, loc="upper left")
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
for ax in axes.flat[len(voices):]:
    ax.axis("off")
for ax in axes[-1]:
    ax.set_xlabel("% of voiced part")
for ax in axes[:, 0]:
    ax.set_ylabel("semitones from start")
fig.suptitle("Pitch contour, median and middle 50% of clips, tone 2 vs tone 3", y=0.995)
fig.tight_layout()
plt.show()

# ---------------------------------------------------------------- 3. pitch-only classifier, training voices only
def clf():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=0.5, max_iter=3000))

tr = t[~t.voice.isin(HELD_OUT)]
SETS = {"shape measures": SHAPE, "whole contour": C + ["creak"]}
models, rows = {}, []
for name, feats in SETS.items():
    models[name] = clf().fit(tr[feats], tr.tone == "2")
    for v in HELD_OUT:
        te = t[t.voice == v]
        pred2 = models[name].predict(te[feats])
        lab2 = (te.tone == "2").values
        m_err = ((te.tone == "2") & (te.heard == "3")).values
        m_ok3 = ((te.tone == "3") & (te.heard == "3")).values
        rows.append({"features": name, "voice": v, "accuracy": f"{(pred2 == lab2).mean():.0%}",
                     "model's 2->3 errors put right": f"{pred2[m_err].sum()}/{m_err.sum()}",
                     "model's correct 3s broken": f"{pred2[m_ok3].sum()}/{m_ok3.sum()}"})
print("\nPitch only, tone 2 vs 3. Trained on the 8 training voices, tested on the 4 held-out (chance = 50%)")
print(pd.DataFrame(rows).pivot(index="voice", columns="features").to_string())

# ---------------------------------------------------------------- 4. pitch re-decides the model's 2-vs-3 split
def evaluate(Pm, idx):
    """all-3 accuracy, tone accuracy, false alarms, with score.py's forgiveness rules."""
    res = pd.DataFrame({"source": meta.loc[idx, "source"]}, index=idx)
    flags = []
    for part in ("initial", "final", "tone"):
        cols = [c for c in Pm.columns if c.startswith(part + ":") and "silence" not in c]
        heard = Pm.loc[idx, cols].idxmax(axis=1).str.split(":").str[1]
        tgt = meta.loc[idx, f"target_{part}"]
        fg = FORGIVE.get(part, {})
        res[f"ok_{part}"] = [(h == g) or (h in fg.get(g, ())) for h, g in zip(heard, tgt)]
        credit = np.array([Pm.at[i, f"{part}:{g}"] + sum(Pm.at[i, f"{part}:{u}"] for u in fg.get(g, ()))
                           for i, g in zip(idx, tgt)])
        flags.append(credit < THR)
    res["all3"] = res[["ok_initial", "ok_final", "ok_tone"]].all(axis=1)
    res["false_alarm"] = np.any(flags, axis=0)
    return res

held_idx = meta.index[meta.source.isin(HELD_OUT)]
have = held_idx.intersection(t.index)                 # tone 2/3 clips with a pitch track
feats = SETS["shape measures"]
q_pitch = pd.Series(models["shape measures"].predict_proba(t.loc[have, feats])[:, 1], index=have)  # P(tone 2)

variants = {"model alone": P}
for name, w in [("pitch decides 2 vs 3", 1.0), ("model and pitch averaged", 0.5)]:
    P2 = P.copy()
    m2, m3 = P2.loc[have, "tone:2"], P2.loc[have, "tone:3"]
    mass = m2 + m3
    lm = np.log(m2 + 1e-9) - np.log(m3 + 1e-9)
    lp = np.log(q_pitch + 1e-6) - np.log(1 - q_pitch + 1e-6)
    q = 1 / (1 + np.exp(-((1 - w) * lm + w * lp)))
    P2.loc[have, "tone:2"], P2.loc[have, "tone:3"] = mass * q, mass * (1 - q)
    variants[name] = P2

rows = []
for vname, Pm in variants.items():
    r = evaluate(Pm, held_idx)
    for v, g in list(r.groupby("source")) + [("BOTH WEB VOICES", r[r.source.isin(["studycli", "uvm"])])]:
        rows.append({"variant": vname, "voice": v, "all-3 %": round(100 * g.all3.mean(), 1),
                     "tone %": round(100 * g.ok_tone.mean(), 1), "false alarms %": round(100 * g.false_alarm.mean(), 1)})
res = pd.DataFrame(rows)
order = list(variants)
print("\n=== 6b+7 with pitch re-deciding tone 2 vs 3 (forgiveness rules from score.py applied)")
print(res.pivot(index="voice", columns="variant", values=["all-3 %", "tone %", "false alarms %"])
      .reindex(columns=order, level=1).to_string())

rows = []
for said, target in [("2", "3"), ("3", "2")]:
    m = meta.loc[held_idx, "target_tone"] == said
    ids = m.index[m]
    r = {"clip says tone": said, "scored as attempt at": target, "clips": len(ids)}
    for vname, Pm in variants.items():
        r[vname] = f"{100 * (Pm.loc[ids, f'tone:{target}'] >= THR).mean():.1f}%"
    rows.append(r)
print("\nReal tone mistakes that would be marked correct (lower is better):")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ============================================================== CELL 11: tone 2 vs 3, version 2 (run cell 10 first)
# Reuses t, P, meta, C, held_idx, have, evaluate, HELD_OUT, clf from cell 10. A few seconds.
# Changes from cell 10:
#  - Dips deeper than 8 semitones are capped. Those are creaky tone 3s that Praat tracked at half
#    pitch (FV2, mandarintemple, yabla); they taught the classifier "tone 3 = huge dip".
#  - Rise and end height are dropped: a full tone 3 also rises at the end (uvm, MV3), so they mislead.
#    What's kept: how deep the dip is, where it is, how much of the syllable sits well below the start,
#    and creak.
#  - A no-training rule: "dip deeper than X semitones = tone 3", X picked on the training voices only.
#  - Pitch can only nudge the model, not overrule it: its vote is capped (variants marked "capped").
import numpy as np, pandas as pd

t2 = t.copy()
inner = t2[C].to_numpy()[:, 2:-2]
t2["dip_c"] = t2.dip.clip(upper=8)
t2["low_frac"] = (inner < -2).mean(axis=1)          # share of the syllable more than 2 st below its start
FEATS2 = ["dip_c", "dip_pos", "low_frac", "creak"]
tr2 = t2[~t2.voice.isin(HELD_OUT)]

# the rule's threshold, chosen on the training voices only
grid = np.arange(0.5, 6.01, 0.1)
acc = [((tr2.dip_c < x) == (tr2.tone == "2")).mean() for x in grid]
X = float(grid[int(np.argmax(acc))])
print(f"rule: dip shallower than {X:.1f} semitones = tone 2  (training-voice accuracy {max(acc):.1%})")

clf2 = clf().fit(tr2[FEATS2], tr2.tone == "2")
logit = lambda p: np.log(p + 1e-6) - np.log(1 - p + 1e-6)
votes = {   # pitch's log-odds for tone 2 (vs 3), per clip
    "v1 features (cell 10)": logit(pd.Series(models["shape measures"].predict_proba(t.loc[have, SHAPE])[:, 1], index=have)),
    "v2 features": logit(pd.Series(clf2.predict_proba(t2.loc[have, FEATS2])[:, 1], index=have)),
    "dip rule": (X - t2.loc[have, "dip_c"]) / 0.7,
}

def blend(vote, w=0.5, cap=None):
    P2 = P.copy()
    m2, m3 = P2.loc[have, "tone:2"], P2.loc[have, "tone:3"]
    lm = np.log(m2 + 1e-9) - np.log(m3 + 1e-9)
    lp = vote.clip(-cap, cap) if cap else vote
    q = 1 / (1 + np.exp(-((1 - w) * lm + w * lp)))
    P2.loc[have, "tone:2"], P2.loc[have, "tone:3"] = (m2 + m3) * q, (m2 + m3) * (1 - q)
    return P2

variants = {"model alone": P,
            "v1 averaged (cell 10)": blend(votes["v1 features (cell 10)"]),
            "v2 averaged": blend(votes["v2 features"]),
            "v2 averaged, capped": blend(votes["v2 features"], cap=4),
            "dip rule averaged": blend(votes["dip rule"]),
            "dip rule averaged, capped": blend(votes["dip rule"], cap=4)}
order = list(variants)

rows = []
for vname, Pm in variants.items():
    r = evaluate(Pm, held_idx)
    for v, g in list(r.groupby("source")) + [("BOTH WEB VOICES", r[r.source.isin(["studycli", "uvm"])])]:
        rows.append({"variant": vname, "voice": v, "all-3 %": round(100 * g.all3.mean(), 1),
                     "tone %": round(100 * g.ok_tone.mean(), 1), "false alarms %": round(100 * g.false_alarm.mean(), 1)})
res = pd.DataFrame(rows)
for col in ["all-3 %", "tone %", "false alarms %"]:
    print(f"\n=== {col}")
    print(res.pivot(index="voice", columns="variant", values=col)[order].to_string())

# what changed, clip by clip, on tone 2/3 clips of the held-out voices
tc = [c for c in P.columns if c.startswith("tone:") and "silence" not in c]
base = P.loc[have, tc].idxmax(axis=1).str.split(":").str[1]
tgt = meta.loc[have, "target_tone"]
rows = []
for vname, Pm in variants.items():
    if vname == "model alone":
        continue
    new = Pm.loc[have, tc].idxmax(axis=1).str.split(":").str[1]
    for v in HELD_OUT:
        m = meta.loc[have, "source"] == v
        rows.append({"variant": vname, "voice": v,
                     "fixed": int(((base != tgt) & (new == tgt) & m).sum()),
                     "broken": int(((base == tgt) & (new != tgt) & m).sum())})
print("\n=== tone 2/3 clips: fixed vs broken compared with the model alone")
print(pd.DataFrame(rows).pivot(index="voice", columns="variant", values=["fixed", "broken"]).to_string())

rows = []
for said, target in [("2", "3"), ("3", "2")]:
    ids = meta.loc[held_idx].index[meta.loc[held_idx, "target_tone"] == said]
    r = {"clip says tone": said, "scored as attempt at": target}
    for vname, Pm in variants.items():
        r[vname] = f"{100 * (Pm.loc[ids, f'tone:{target}'] >= THR).mean():.1f}%"
    rows.append(r)
print("\n=== real tone mistakes that would be marked correct (lower is better)")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
rows = []
for w in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    Pm = blend(votes["v1 features (cell 10)"], w=w)
    r = evaluate(Pm, held_idx)
    web = r[r.source.isin(["studycli", "uvm"])]
    ids2 = meta.loc[held_idx].index[meta.loc[held_idx, "target_tone"] == "3"]
    rows.append({"pitch weight": w,
                 "web all-3 %": round(100 * web.all3.mean(), 1),
                 "studycli all-3 %": round(100 * r[r.source == "studycli"].all3.mean(), 1),
                 "uvm all-3 %": round(100 * r[r.source == "uvm"].all3.mean(), 1),
                 "web false alarms %": round(100 * web.false_alarm.mean(), 1),
                 "tone 3 passes as tone 2 %": round(100 * (Pm.loc[ids2, "tone:2"] >= THR).mean(), 1)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
rows = []
for w in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    Pm = blend(votes["v1 features (cell 10)"], w=w)
    r = evaluate(Pm, held_idx)
    web = r[r.source.isin(["studycli", "uvm"])]
    ids2 = meta.loc[held_idx].index[meta.loc[held_idx, "target_tone"] == "3"]
    rows.append({"pitch weight": w,
                 "web all-3 %": round(100 * web.all3.mean(), 1),
                 "studycli all-3 %": round(100 * r[r.source == "studycli"].all3.mean(), 1),
                 "uvm all-3 %": round(100 * r[r.source == "uvm"].all3.mean(), 1),
                 "web false alarms %": round(100 * web.false_alarm.mean(), 1),
                 "tone 3 passes as tone 2 %": round(100 * (Pm.loc[ids2, "tone:2"] >= THR).mean(), 1)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# ============================================================== CELL 12: save the pitch classifier for score.py
# Needs cell 10 to have run once (it saved pitch_tone23.csv on Drive), the session setup, and the new
# pitch_tone.py + score.py in the repo. A few seconds.
import importlib, json
import joblib, numpy as np, pandas as pd
import pitch_tone; importlib.reload(pitch_tone)
from data import load_audio
from pinyin_units import split_syllable

runs = "/content/drive/MyDrive/pinyin_runs"
TRAIN_MANIFESTS = ["manifests/syl_train.jsonl", "manifests/good/train.jsonl"]   # training voices only
OUT = f"{runs}/tone23_pitch.joblib"
C = [f"c{i}" for i in range(pitch_tone.N)]

rows = []
for m in TRAIN_MANIFESTS:
    for line in open(m):
        r = json.loads(line)
        p = split_syllable(r["pinyin"])
        if p and str(p[2]) in ("2", "3"):
            rows.append({"audio": r["audio"], "voice": r.get("speaker") or r.get("source"), "tone": str(p[2])})
clips = pd.DataFrame(rows).drop_duplicates("audio").set_index("audio")
pc = pd.read_csv(f"{runs}/pitch_tone23.csv").set_index("audio")
t = clips.join(pc, how="inner")
t = t[t.c0.notna()]

# check that pitch_tone.py measures exactly what cell 10 saved (same code, so this should be ~0)
sample = t.sample(20, random_state=0)
diffs = []
for a, r in sample.iterrows():
    got = pitch_tone.contour(load_audio(a))
    if got is not None:
        diffs.append(np.abs(got[0] - r[C].to_numpy(dtype=float)).max())
print(f"pitch_tone.py vs cell 10 contours, largest difference on 20 clips: {max(diffs):.4f} semitones")

F = pd.DataFrame([pitch_tone.features_from_contour(r[C].to_numpy(dtype=float), r.creak) for _, r in t.iterrows()],
                 index=t.index)
clf = pitch_tone.make_classifier().fit(F[pitch_tone.SHAPE].to_numpy(), (t.tone == "2").to_numpy())
print(f"trained on {len(t)} clips from {sorted(t.voice.unique())}")
print(f"training accuracy {(clf.predict(F[pitch_tone.SHAPE].to_numpy()) == (t.tone == '2').to_numpy()).mean():.1%}")
joblib.dump({"clf": clf, "features": pitch_tone.SHAPE, "trained_on": sorted(t.voice.unique())}, OUT)
print("saved", OUT)

In [ ]:
# ============================================================== CELL 13: how much do the score.py updates change accuracy?
# Runs the real score.py (two models + forgiveness + pitch) on every test clip. ~6-10 minutes on the GPU
# the first time; the raw outputs are cached on Drive, so reruns take seconds.
# Compares four set-ups, all on the same clips:
#   old score.py: 6b alone | 6b+7 | 6b+7 + forgiveness | 6b+7 + forgiveness + pitch  (= new score.py)
import importlib, json, os, pickle, random, time
import numpy as np, pandas as pd
import pitch_tone, score
importlib.reload(pitch_tone); importlib.reload(score)
from score import Scorer, combine, apply_pitch, judge, credited, FORGIVE, PITCH_WEIGHT, DEFAULT_THRESHOLD as THR
from pinyin_units import HEADS, split_syllable

runs = "/content/drive/MyDrive/pinyin_runs"
MODELS = [f"{runs}/stage6b/best", f"{runs}/stage7_hubert/best"]
PITCH = f"{runs}/tone23_pitch.joblib"
TEST = ["manifests/syl_test.jsonl", "manifests/good/test_studycli.jsonl", "manifests/good/test_uvm.jsonl"]
CACHE = f"{runs}/eval_cache_6b_7_pitch.pkl"
WEB = ["studycli", "uvm"]

clips = []
for m in TEST:
    for line in open(m):
        r = json.loads(line)
        clips.append({"audio": r["audio"], "voice": r.get("speaker") or r.get("source"), "pinyin": r["pinyin"]})
print(f"{len(clips)} test clips")

scorer = Scorer(MODELS, pitch_model=PITCH)
cache = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}
todo = [c for c in clips if c["audio"] not in cache]
for i, c in enumerate(todo, 1):
    wav = Scorer.prepare(c["audio"])
    cache[c["audio"]] = {"mp": scorer.model_probs(wav), "vote": scorer.pitch_vote(wav)}
    if i % 500 == 0:
        print(f"  {i}/{len(todo)}", flush=True)
        pickle.dump(cache, open(CACHE, "wb"))
if todo:
    pickle.dump(cache, open(CACHE, "wb"))

SETUPS = {
    "old: 6b alone": lambda x: (combine(x["mp"][:1]), {}),
    "6b+7": lambda x: (combine(x["mp"]), {}),
    "6b+7 + forgiveness": lambda x: (combine(x["mp"]), FORGIVE),
    "NEW: + pitch": lambda x: (apply_pitch(combine(x["mp"]), x["vote"], PITCH_WEIGHT), FORGIVE),
}

rows = []
for c in clips:
    x = cache[c["audio"]]
    for name, make in SETUPS.items():
        probs, fg = make(x)
        res = judge(probs, c["pinyin"], THR, forgive=fg)
        ok = {h: (p["heard_unit"] == p["target_unit"]) or p["forgiven"] for h, p in res["parts"].items()}
        rows.append({"voice": c["voice"], "setup": name, "all3": all(ok.values()), "tone": ok["tone"],
                     "final": ok["final"], "false_alarm": not res["understood"]})
R = pd.DataFrame(rows)
order = list(SETUPS)

def table(col):
    g = R.groupby(["voice", "setup"])[col].mean().unstack("setup")
    g.loc["BOTH WEB VOICES"] = R[R.voice.isin(WEB)].groupby("setup")[col].mean()
    return (100 * g[order]).round(1)

for col, title in [("all3", "all-3 accuracy %"), ("false_alarm", "false alarms %"),
                   ("tone", "tone accuracy %"), ("final", "final accuracy %")]:
    print(f"\n=== {title}")
    print(table(col).to_string())

# missed mistakes: native clip scored as if the learner was attempting something else
SWAPS = [("final", "ang", "eng"), ("final", "eng", "ang"), ("final", "an", "en"), ("final", "en", "an"),
         ("initial", "x", "q"), ("initial", "q", "x"), ("initial", "zh", "j"), ("initial", "j", "zh"),
         ("tone", "2", "3"), ("tone", "3", "2")]
parts = {c["audio"]: dict(zip(HEADS, split_syllable(c["pinyin"]))) for c in clips}
rows = []
for head, said, target in SWAPS:
    ids = [c["audio"] for c in clips if str(parts[c["audio"]][head]) == said]
    if not ids:
        continue
    r = {"part": head, "clip says": said, "scored as attempt at": target, "clips": len(ids)}
    for name, make in SETUPS.items():
        passed = []
        for a in ids:
            probs, fg = make(cache[a])
            passed.append(credited(probs, head, target, fg) >= THR)
        r[name] = f"{100 * np.mean(passed):.1f}%"
    rows.append(r)
print("\n=== real mistakes that would be marked correct (lower is better)")
print(pd.DataFrame(rows).to_string(index=False))

# check: the real score.py gives the same verdict as the NEW column, and how long it takes
random.seed(0)
check = random.sample(clips, 40)
t0 = time.time()
same = sum(scorer.score(c["audio"], c["pinyin"])["understood"] ==
           judge(SETUPS["NEW: + pitch"](cache[c["audio"]])[0], c["pinyin"], THR)["understood"] for c in check)
ms = 1000 * (time.time() - t0) / len(check)
print(f"\nscore.py agrees with the NEW column on {same}/{len(check)} clips; {ms:.0f} ms per call "
      f"(2 models + pitch, this GPU)")
pitch_found = np.mean([cache[c["audio"]]["vote"] is not None for c in clips])
print(f"pitch found on {pitch_found:.1%} of clips (no pitch = model only)")

In [ ]:
# ============================================================== CELL 14: should creaky voice count as tone 3?
# Needs the session setup (audio unzipped), the new pitch_tone.py in the repo, and cell 13's cache on Drive.
# ~4 minutes the first time (creak measured on every clip, all voices, all tones); cached after.
# Proposed rule: when the TARGET is tone 3 and the middle of the syllable is creaky, the tone passes.
# It can only remove false alarms on tone 3 targets. Its cost: a learner who says tone 1, 2 or 4 with
# creak, when tone 3 was asked for, would also pass. This cell measures both.
import importlib, json, os, pickle
import numpy as np, pandas as pd
from multiprocessing import Pool
import pitch_tone, score
importlib.reload(pitch_tone); importlib.reload(score)
from score import combine, apply_pitch, judge, credited, FORGIVE, PITCH_WEIGHT, DEFAULT_THRESHOLD as THR
from pinyin_units import split_syllable

runs = "/content/drive/MyDrive/pinyin_runs"
MANIFESTS = ["manifests/syl_train.jsonl", "manifests/good/train.jsonl",
             "manifests/syl_test.jsonl", "manifests/good/test_studycli.jsonl", "manifests/good/test_uvm.jsonl"]
CREAK_CSV = f"{runs}/creak_all.csv"
CACHE = f"{runs}/eval_cache_6b_7_pitch.pkl"          # from cell 13
HELD_OUT = ["FV3", "MV3", "studycli", "uvm"]
WEB = ["studycli", "uvm"]
THRESHOLDS = [0.1, 0.2, 0.3, 0.4]

def measure(audio):
    from data import load_audio
    try:
        return {"audio": audio, "creak": pitch_tone.creak_fraction(load_audio(audio))}
    except Exception:
        return {"audio": audio, "creak": None}

rows = []
for m in MANIFESTS:
    for line in open(m):
        r = json.loads(line)
        p = split_syllable(r["pinyin"])
        if p:
            rows.append({"audio": r["audio"], "voice": r.get("speaker") or r.get("source"),
                         "pinyin": r["pinyin"], "tone": str(p[2])})
clips = pd.DataFrame(rows).drop_duplicates("audio").set_index("audio")
if os.path.exists(CREAK_CSV):
    cr = pd.read_csv(CREAK_CSV).set_index("audio")
else:
    with Pool(2) as pool:
        cr = pd.DataFrame(pool.map(measure, list(clips.index), chunksize=32)).set_index("audio")
    cr.to_csv(CREAK_CSV)
c = clips.join(cr)
print(f"creak measured on {c.creak.notna().mean():.1%} of {len(c)} clips")

# ---------------------------------------------------------------- 1. how creaky is each tone, per voice?
for th in (0.2, 0.3):
    tab = c.assign(creaky=c.creak >= th).groupby(["voice", "tone"]).creaky.mean().unstack("tone")
    tab = (100 * tab[[t for t in ["1", "2", "3", "4", "5"] if t in tab.columns]]).round(1)
    tab.index = [v + (" (held out)" if v in HELD_OUT else "") for v in tab.index]
    print(f"\n=== % of clips with creak >= {th:.0%} of the syllable's middle, by tone")
    print(tab.to_string())

# ---------------------------------------------------------------- 2. effect on the current score.py
cache = pickle.load(open(CACHE, "rb"))
test = c[c.voice.isin(HELD_OUT) & c.index.isin(list(cache))]
probs = {a: apply_pitch(combine(cache[a]["mp"]), cache[a]["vote"], PITCH_WEIGHT) for a in test.index}
base = {a: judge(probs[a], test.at[a, "pinyin"], THR) for a in test.index}

rows = []
for th in [None] + THRESHOLDS:
    fa = {}
    for a, r in test.iterrows():
        res = base[a]
        tone_ok = res["parts"]["tone"]["correct"] or (
            th is not None and r.tone == "3" and r.creak is not None and not np.isnan(r.creak) and r.creak >= th)
        others_ok = res["parts"]["initial"]["correct"] and res["parts"]["final"]["correct"]
        fa[a] = res["no_speech"] or not (tone_ok and others_ok)
    s = pd.Series(fa)
    row = {"rule": "current score.py" if th is None else f"creak >= {th:.0%} counts as tone 3"}
    for v in HELD_OUT:
        row[v] = round(100 * s[test.voice == v].mean(), 2)
    row["BOTH WEB VOICES"] = round(100 * s[test.voice.isin(WEB)].mean(), 2)
    row["tone 3 clips flagged"] = int(sum(fa[a] for a in test.index[test.tone == "3"]))
    rows.append(row)
print("\n=== false alarms % (current score.py, with and without the creak rule)")
print(pd.DataFrame(rows).to_string(index=False))

# cost: native tone 1/2/4 clips scored as if the learner was asked for tone 3
rows = []
for said in ["1", "2", "4"]:
    ids = test.index[test.tone == said]
    if len(ids) == 0:
        continue
    before = np.array([credited(probs[a], "tone", "3") >= THR for a in ids], dtype=bool)
    row = {"clip says tone": said, "scored as attempt at": "3", "clips": len(ids),
           "current score.py": f"{100 * before.mean():.1f}%"}
    for th in THRESHOLDS:
        cre = np.array([(x is not None and not np.isnan(x) and x >= th) for x in test.loc[ids, "creak"]], dtype=bool)
        row[f"creak >= {th:.0%}"] = f"{100 * (before | cre).mean():.1f}%"
    rows.append(row)
print("\n=== cost: real mistakes that would be marked correct as tone 3 (lower is better)")
print(pd.DataFrame(rows).to_string(index=False))

# ---------------------------------------------------------------- 3. listen: creaky clips of each tone
from IPython.display import Audio, display
from data import load_audio
LISTEN_TH, PER_TONE = 0.3, 2
for tone in ["3", "4", "2", "1"]:
    g = c[(c.tone == tone) & (c.creak >= LISTEN_TH)].sort_values("creak", ascending=False).head(PER_TONE)
    for a, r in g.iterrows():
        print(f"tone {tone}  {r.voice:14} {r.pinyin:8} creak {r.creak:.0%}")
        display(Audio(load_audio(a), rate=16000))

In [ ]:
!grep -c "def creak_fraction" /content/repo/pinyin_scorer/pitch_tone.py    # should print 1
!rm -f /content/drive/MyDrive/pinyin_runs/creak_all.csv                   # clear the empty cache

# try one clip directly, so any error is shown
import importlib, pitch_tone; importlib.reload(pitch_tone)
from data import load_audio
import json
a = json.loads(open("manifests/good/test_studycli.jsonl").readline())["audio"]
print(pitch_tone.creak_fraction(load_audio(a)))

In [ ]:
# ============================================================== CELL 15: accept a half third tone when tone 3 is the target
# Needs the session setup, the new score.py + pitch_tone.py, tone23_pitch.joblib, and cell 13's cache.
# ~6 minutes (pitch on ~18k clips on the CPU, then ~1,500 clips through the two models on the GPU).
#
# The data only has full third tones (dip + rise). In speech a tone 3 is often a "half third": it
# falls low and stays there, with no rise. There are no such clips, so this cell MAKES them: every
# native tone 3 clip is cut off at its lowest pitch point, removing the rise. Listen to them first.
#
# A half third and a tone 4 both fall with no rise, so the danger is accepting a tone 4 for a tone 3
# target. A small pitch classifier (trained on the TRAINING voices only) learns half third vs native
# tones 1, 2 and 4;
# the rule only applies when the target is tone 3, and is measured on the held-out voices for both:
#   gain: half thirds accepted         cost: native tone 4 (and 1, 2) accepted for a tone 3 target
import importlib, json, os, pickle, warnings
import numpy as np, pandas as pd
from multiprocessing import Pool
import pitch_tone, score
importlib.reload(pitch_tone); importlib.reload(score)
from score import Scorer, combine, apply_pitch, judge, credited, PITCH_WEIGHT, DEFAULT_THRESHOLD as THR
from pinyin_units import split_syllable

runs = "/content/drive/MyDrive/pinyin_runs"
MANIFESTS = ["manifests/syl_train.jsonl", "manifests/good/train.jsonl",
             "manifests/syl_test.jsonl", "manifests/good/test_studycli.jsonl", "manifests/good/test_uvm.jsonl"]
CACHE = f"{runs}/eval_cache_6b_7_pitch.pkl"          # from cell 13 (native test clips)
HELD_OUT = ["FV3", "MV3", "studycli", "uvm"]
SR = 16000
C = [f"c{i}" for i in range(pitch_tone.N)]

def cut_at_turn(wav):
    """The clip cut 30 ms after its lowest pitch point, i.e. without the rise. None if no clear dip-then-rise."""
    import parselmouth
    from scipy.signal import medfilt
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        snd = parselmouth.Sound(wav.astype(np.float64), sampling_frequency=SR)
        f = snd.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=600).selected_array["frequency"]
        if (f > 0).sum() < 8:
            return None
        med = float(np.median(f[f > 0]))
        p = snd.to_pitch(time_step=0.01, pitch_floor=max(60, med / 2.2), pitch_ceiling=min(800, med * 2.2))
        f, t = p.selected_array["frequency"], p.xs()
        v = np.where(f > 0)[0]
        if len(v) < 8:
            return None
        a, b = v[0], v[-1]
        seg = f[a:b + 1].astype(float)
        ok = seg > 0
        idx = np.arange(len(seg))
        st = np.interp(idx, idx[ok], 12 * np.log2(seg[ok] / med))
        st = medfilt(st, 5)
        lo, hi = int(0.15 * len(st)), int(0.9 * len(st))
        i = lo + int(np.argmin(st[lo:hi]))
        if i < 0.2 * len(st) or i > 0.85 * len(st) or st[-3:].mean() - st[i] < 2:
            return None                                  # no clear dip followed by a rise
        end = int((t[a + i] + 0.03) * SR)
        w = wav[:end].copy()
        fade = min(240, len(w))
        w[-fade:] *= np.linspace(1, 0, fade)
        return np.concatenate([w, np.zeros(int(0.15 * SR), np.float32)]).astype(np.float32)

def work(item):
    audio, voice, tone, keep = item
    from data import load_audio
    out = {"audio": audio, "voice": voice, "tone": tone}
    try:
        wav = load_audio(audio)
        if tone == "3":
            wav = cut_at_turn(wav)
            if wav is None:
                return out
        got = pitch_tone.contour(wav)
        if got is None:
            return out
        out.update({k: v for k, v in zip(C, got[0])}, creak=got[1])
        if keep:
            out["wav"] = wav
    except Exception:
        pass
    return out

items = []
for m in MANIFESTS:
    for line in open(m):
        r = json.loads(line)
        p = split_syllable(r["pinyin"])
        v = r.get("speaker") or r.get("source")
        if p and str(p[2]) in ("1", "2", "3", "4"):
            items.append((r["audio"], v, str(p[2]), v in HELD_OUT and str(p[2]) == "3"))
with Pool(2) as pool:
    res = pool.map(work, items, chunksize=32)
wavs = {r["audio"]: r.pop("wav") for r in res if "wav" in r}
D = pd.DataFrame(res).set_index("audio")
D = D[D.c0.notna()]
D["group"] = D.tone.map({"3": "half third (cut tone 3)", "4": "tone 4", "1": "tone 1", "2": "tone 2"})
pin = {it[0]: None for it in items}
for m in MANIFESTS:
    for line in open(m):
        r = json.loads(line)
        if r["audio"] in pin:
            pin[r["audio"]] = r["pinyin"]
print(D.groupby(["group"]).size().to_string())

# ---------------------------------------------------------------- 1. listen first: does the cut sound like a half third?
from IPython.display import Audio, display
from data import load_audio
for a in list(wavs)[:6]:
    print(f"{D.at[a, 'voice']:9} {pin[a]:8}   full clip, then cut:")
    display(Audio(load_audio(a), rate=SR)); display(Audio(wavs[a], rate=SR))

# ---------------------------------------------------------------- 2. what does the current score.py do with half thirds?
scorer = Scorer([f"{runs}/stage6b/best", f"{runs}/stage7_hubert/best"], pitch_model=f"{runs}/tone23_pitch.joblib")
hp = {a: scorer.probs(w) for a, w in wavs.items()}
tones = [k for k in next(iter(hp.values()))["tone"] if not k.startswith("<")]
rows = []
for a, p in hp.items():
    heard = max(tones, key=lambda k: p["tone"][k])
    rows.append({"voice": D.at[a, "voice"], "tone 3 passes": credited(p, "tone", "3") >= THR, "heard": heard})
H = pd.DataFrame(rows)
print("\n=== half thirds (held-out voices) with the current score.py: does the tone pass?")
print(H.groupby("voice").agg(clips=("heard", "size"), passes=("tone 3 passes", "mean"),
                             heard_as=("heard", lambda s: s.value_counts().head(3).to_dict())).to_string())

# ---------------------------------------------------------------- 3. half third vs tones 1, 2, 4, from pitch, training voices only
clf = pitch_tone.make_classifier().fit(D.loc[~D.voice.isin(HELD_OUT), C + ["creak"]].to_numpy(),
                                       (D.loc[~D.voice.isin(HELD_OUT), "tone"] == "3").to_numpy())
D["p_half3"] = clf.predict_proba(D[C + ["creak"]].to_numpy())[:, 1]

cache = pickle.load(open(CACHE, "rb"))
native = {}
for m in MANIFESTS[2:]:
    for line in open(m):
        r = json.loads(line)
        p = split_syllable(r["pinyin"])
        if p and r["audio"] in cache:
            native[r["audio"]] = (r.get("speaker") or r.get("source"), str(p[2]))
nprobs = {a: apply_pitch(combine(cache[a]["mp"]), cache[a]["vote"], PITCH_WEIGHT) for a in native}
cost_ids = [a for a, (v, t) in native.items() if t in ("1", "2", "4") and a in D.index]
cost_p = {a: D.at[a, "p_half3"] for a in cost_ids}
rows = []
for tau in [None, 0.5, 0.7, 0.8, 0.9, 0.95]:
    r = {"rule": "current score.py" if tau is None else f"half third accepted if p >= {tau}"}
    for v in HELD_OUT:
        ids = [a for a in hp if D.at[a, "voice"] == v]
        ok = [credited(hp[a], "tone", "3") >= THR or (tau is not None and D.at[a, "p_half3"] >= tau) for a in ids]
        r[f"half thirds pass: {v}"] = f"{100 * np.mean(ok):.0f}%"
    for said in ["4", "1", "2"]:
        ids = [a for a in cost_ids if native[a][1] == said]
        ok = [credited(nprobs[a], "tone", "3") >= THR or (tau is not None and cost_p[a] >= tau) for a in ids]
        r[f"tone {said} accepted for tone 3"] = f"{100 * np.mean(ok):.1f}%"
    rows.append(r)
print("\n=== the rule, only when the target is tone 3 (held-out voices). Gain = half thirds passing;")
print("    cost = native tone 4/1/2 clips that would pass if the learner was asked for tone 3")
print(pd.DataFrame(rows).set_index("rule").T.to_string())

In [ ]:
# ============================================================== CELL 16: half third tone, version 2 (rise flattened, not cut off)
# Needs the session setup, the new score.py + pitch_tone.py, tone23_pitch.joblib, and cell 13's cache.
# ~6 minutes (pitch on ~18k clips on the CPU, then ~1,500 clips through the two models on the GPU).
#
# The data only has full third tones (dip + rise). In speech a tone 3 is often a "half third": it
# falls low and stays there, with no rise. There are no such clips, so this cell MAKES them: in every
# native tone 3 clip the pitch is held low from its lowest point to the end (Praat resynthesis), so the
# rise is gone but the length and the whole final (e.g. the -i of ai3) are kept. Cell 15 cut the clip
# off instead, which also cut the final and left no low stretch, so its half thirds looked like a short
# tone 4. Listen to them first.
#
# A half third and a tone 4 both fall with no rise, so the danger is accepting a tone 4 for a tone 3
# target. A small pitch classifier (trained on the TRAINING voices only) learns half third vs native
# tones 1, 2 and 4;
# the rule only applies when the target is tone 3, and is measured on the held-out voices for both:
#   gain: half thirds accepted         cost: native tone 4 (and 1, 2) accepted for a tone 3 target
import importlib, json, os, pickle, warnings
import numpy as np, pandas as pd
from multiprocessing import Pool
import pitch_tone, score
importlib.reload(pitch_tone); importlib.reload(score)
from score import Scorer, combine, apply_pitch, judge, credited, PITCH_WEIGHT, DEFAULT_THRESHOLD as THR
from pinyin_units import split_syllable

runs = "/content/drive/MyDrive/pinyin_runs"
MANIFESTS = ["manifests/syl_train.jsonl", "manifests/good/train.jsonl",
             "manifests/syl_test.jsonl", "manifests/good/test_studycli.jsonl", "manifests/good/test_uvm.jsonl"]
CACHE = f"{runs}/eval_cache_6b_7_pitch.pkl"          # from cell 13 (native test clips)
HELD_OUT = ["FV3", "MV3", "studycli", "uvm"]
SR = 16000
C = [f"c{i}" for i in range(pitch_tone.N)]

def flatten_rise(wav):
    """The same clip with the rise removed: after its lowest pitch point the pitch stays low (drifting
    down 1 semitone) to the end, resynthesised with Praat. Length and the whole final are kept.
    None if there is no clear dip followed by a rise."""
    import parselmouth
    from parselmouth.praat import call
    from scipy.signal import medfilt
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        snd = parselmouth.Sound(wav.astype(np.float64), sampling_frequency=SR)
        f = snd.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=600).selected_array["frequency"]
        if (f > 0).sum() < 8:
            return None
        med = float(np.median(f[f > 0]))
        floor, ceil = max(60, med / 2.2), min(800, med * 2.2)
        p = snd.to_pitch(time_step=0.01, pitch_floor=floor, pitch_ceiling=ceil)
        f, t = p.selected_array["frequency"], p.xs()
        v = np.where(f > 0)[0]
        if len(v) < 8:
            return None
        a, b = v[0], v[-1]
        seg = f[a:b + 1].astype(float)
        ok = seg > 0
        idx = np.arange(len(seg))
        st = np.interp(idx, idx[ok], 12 * np.log2(seg[ok] / med))
        st = medfilt(st, 5)
        lo, hi = int(0.15 * len(st)), int(0.9 * len(st))
        i = lo + int(np.argmin(st[lo:hi]))
        if i < 0.2 * len(st) or i > 0.85 * len(st) or st[-3:].mean() - st[i] < 2:
            return None                                  # no clear dip followed by a rise
        t_min, t_end, f_min = float(t[a + i]), float(t[b]), med * 2 ** (st[i] / 12)
        manip = call(snd, "To Manipulation", 0.01, floor, ceil)
        tier = call(manip, "Extract pitch tier")
        call(tier, "Remove points between", t_min, snd.duration)
        call(tier, "Add point", t_min, f_min)
        call(tier, "Add point", t_end, f_min * 2 ** (-1 / 12))
        call([manip, tier], "Replace pitch tier")
        out = call(manip, "Get resynthesis (overlap-add)")
        return np.asarray(out.values[0], dtype=np.float32)


def work(item):
    audio, voice, tone, keep = item
    from data import load_audio
    out = {"audio": audio, "voice": voice, "tone": tone}
    try:
        wav = load_audio(audio)
        if tone == "3":
            wav = flatten_rise(wav)
            if wav is None:
                return out
        got = pitch_tone.contour(wav)
        if got is None:
            return out
        out.update({k: v for k, v in zip(C, got[0])}, creak=got[1])
        if keep:
            out["wav"] = wav
    except Exception:
        pass
    return out

items = []
for m in MANIFESTS:
    for line in open(m):
        r = json.loads(line)
        p = split_syllable(r["pinyin"])
        v = r.get("speaker") or r.get("source")
        if p and str(p[2]) in ("1", "2", "3", "4"):
            items.append((r["audio"], v, str(p[2]), v in HELD_OUT and str(p[2]) == "3"))
with Pool(2) as pool:
    res = pool.map(work, items, chunksize=32)
wavs = {r["audio"]: r.pop("wav") for r in res if "wav" in r}
D = pd.DataFrame(res).set_index("audio")
D = D[D.c0.notna()]
D["group"] = D.tone.map({"3": "half third (tone 3, rise flattened)", "4": "tone 4", "1": "tone 1", "2": "tone 2"})
pin = {it[0]: None for it in items}
for m in MANIFESTS:
    for line in open(m):
        r = json.loads(line)
        if r["audio"] in pin:
            pin[r["audio"]] = r["pinyin"]
print(D.groupby(["group"]).size().to_string())

# ---------------------------------------------------------------- 1. listen first: does the cut sound like a half third?
from IPython.display import Audio, display
from data import load_audio
for a in list(wavs)[:6]:
    print(f"{D.at[a, 'voice']:9} {pin[a]:8}   full clip, then half third:")
    display(Audio(load_audio(a), rate=SR)); display(Audio(wavs[a], rate=SR))

# ---------------------------------------------------------------- 2. what does the current score.py do with half thirds?
scorer = Scorer([f"{runs}/stage6b/best", f"{runs}/stage7_hubert/best"], pitch_model=f"{runs}/tone23_pitch.joblib")
hp = {a: scorer.probs(w) for a, w in wavs.items()}
tones = [k for k in next(iter(hp.values()))["tone"] if not k.startswith("<")]
rows = []
for a, p in hp.items():
    heard = max(tones, key=lambda k: p["tone"][k])
    rows.append({"voice": D.at[a, "voice"], "tone 3 passes": credited(p, "tone", "3") >= THR, "heard": heard})
H = pd.DataFrame(rows)
print("\n=== half thirds (held-out voices) with the current score.py: does the tone pass?")
print(H.groupby("voice").agg(clips=("heard", "size"), passes=("tone 3 passes", "mean"),
                             heard_as=("heard", lambda s: s.value_counts().head(3).to_dict())).to_string())

# ---------------------------------------------------------------- 3. half third vs tones 1, 2, 4, from pitch, training voices only
clf = pitch_tone.make_classifier().fit(D.loc[~D.voice.isin(HELD_OUT), C + ["creak"]].to_numpy(),
                                       (D.loc[~D.voice.isin(HELD_OUT), "tone"] == "3").to_numpy())
D["p_half3"] = clf.predict_proba(D[C + ["creak"]].to_numpy())[:, 1]

cache = pickle.load(open(CACHE, "rb"))
native = {}
for m in MANIFESTS[2:]:
    for line in open(m):
        r = json.loads(line)
        p = split_syllable(r["pinyin"])
        if p and r["audio"] in cache:
            native[r["audio"]] = (r.get("speaker") or r.get("source"), str(p[2]))
nprobs = {a: apply_pitch(combine(cache[a]["mp"]), cache[a]["vote"], PITCH_WEIGHT) for a in native}
cost_ids = [a for a, (v, t) in native.items() if t in ("1", "2", "4") and a in D.index]
cost_p = {a: D.at[a, "p_half3"] for a in cost_ids}
rows = []
for tau in [None, 0.5, 0.7, 0.8, 0.9, 0.95]:
    r = {"rule": "current score.py" if tau is None else f"half third accepted if p >= {tau}"}
    for v in HELD_OUT:
        ids = [a for a in hp if D.at[a, "voice"] == v]
        ok = [credited(hp[a], "tone", "3") >= THR or (tau is not None and D.at[a, "p_half3"] >= tau) for a in ids]
        r[f"half thirds pass: {v}"] = f"{100 * np.mean(ok):.0f}%"
    for said in ["4", "1", "2"]:
        ids = [a for a in cost_ids if native[a][1] == said]
        ok = [credited(nprobs[a], "tone", "3") >= THR or (tau is not None and cost_p[a] >= tau) for a in ids]
        r[f"tone {said} accepted for tone 3"] = f"{100 * np.mean(ok):.1f}%"
    rows.append(r)
print("\n=== the rule, only when the target is tone 3 (held-out voices). Gain = half thirds passing;")
print("    cost = native tone 4/1/2 clips that would pass if the learner was asked for tone 3")
print(pd.DataFrame(rows).set_index("rule").T.to_string())

In [25]:
runs = "/content/drive/MyDrive/pinyin_runs"
!mkdir -p /content/models
!cp -r {runs}/stage6b/best /content/models/stage6b
!cp -r {runs}/stage7_hubert/best /content/models/stage7_hubert
!cp {runs}/tone23_pitch.joblib /content/models/
!du -sh /content/models && cd /content && zip -qr /content/drive/MyDrive/pinyin_models.zip models
!pip freeze | grep -iE "^(torch|transformers|scikit-learn|praat-parselmouth|numpy|scipy)="

721M	/content/models
numpy==2.1.3
praat-parselmouth==0.4.7
scikit-learn==1.6.1
scipy==1.16.3
torch==2.11.0+cu130
transformers==5.18.0
